# 롤링 코호트 피처 엔지니어링

## 목적

미식 방문형 롤링 코호트의 사용자-연도 표본에 대해
선정연도와 관찰연도의 리뷰 활동 변화를 피처로 생성한다.

## 시간 구조

- 선정연도: baseline
- 다음 연도: recent
- 다다음 연도: churn 타깃

## 분석 단위

user_id × selection_year

## 주의

타깃 연도의 리뷰 정보는 피처 생성에 사용하지 않는다

In [1]:
# 1. 라이브러리 및 프로젝트 경로
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
current_path = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        path
        for path in [
            current_path,
            *current_path.parents
        ]
        if (
            path
            / "data"
            / "interim"
        ).exists()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "프로젝트 루트를 찾을 수 없습니다."
    )

INTERIM_DIR = (
    PROJECT_ROOT
    / "data"
    / "interim"
)

ROLLING_COHORT_DIR = (
    INTERIM_DIR
    / "rolling"
)

ROLLING_FEATURE_DIR = (
    INTERIM_DIR
    / "features_rolling"
)

VALIDATION_CACHE_DIR = (
    INTERIM_DIR
    / "validation"
)

REPORT_TABLE_DIR = (
    PROJECT_ROOT
    / "reports"
    / "tables"
)

ROLLING_FEATURE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

REPORT_TABLE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("프로젝트 루트:", PROJECT_ROOT)

프로젝트 루트: C:\Users\playdata2\SKN34-2nd-5Team


In [3]:
# 2. 입력·출력 파일 경로
MASTER_COHORT_PATH = (
    ROLLING_COHORT_DIR
    / "culinary_rolling_cohort_master_v02.parquet"
)

RESTAURANT_MONTHLY_PATH = (
    VALIDATION_CACHE_DIR
    / "restaurant_monthly_activity_validation_v01.parquet"
)

CULINARY_MONTHLY_PATH = (
    VALIDATION_CACHE_DIR
    / "culinary_visit_monthly_activity_v01.parquet"
)

ACTIVITY_FEATURE_PATH = (
    ROLLING_FEATURE_DIR
    / "activity_features_rolling_v02.parquet"
)

print(
    "마스터 코호트:",
    MASTER_COHORT_PATH.exists()
)

print(
    "Restaurants 월간 데이터:",
    RESTAURANT_MONTHLY_PATH.exists()
)

print(
    "미식 방문형 월간 데이터:",
    CULINARY_MONTHLY_PATH.exists()
)

assert MASTER_COHORT_PATH.exists()
assert RESTAURANT_MONTHLY_PATH.exists()
assert CULINARY_MONTHLY_PATH.exists()

print("입력 파일 검증 통과")

마스터 코호트: True
Restaurants 월간 데이터: True
미식 방문형 월간 데이터: True
입력 파일 검증 통과


In [4]:
# 3. 마스터 롤링 코호트 불러오기
master_cohort_df = pd.read_parquet(
    MASTER_COHORT_PATH
)

print(
    "마스터 코호트 크기:",
    master_cohort_df.shape
)

print(
    "고유 표본:",
    master_cohort_df[
        "sample_id"
    ].nunique()
)

print(
    "고유 사용자:",
    master_cohort_df[
        "user_id"
    ].nunique()
)

print(
    "선정연도:",
    sorted(
        master_cohort_df[
            "selection_year"
        ].unique()
    )
)

print(
    "이탈 분포:"
)

print(
    master_cohort_df[
        "churn"
    ].value_counts()
)

마스터 코호트 크기: (21601, 13)
고유 표본: 21601
고유 사용자: 12709
선정연도: [np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017)]
이탈 분포:
churn
0    18215
1     3386
Name: count, dtype: int64


In [5]:
required_cohort_columns = {
    "sample_id",
    "user_id",
    "selection_year",
    "observation_year",
    "target_year",
    "churn",
    "candidate_range_split",
    "recent_range_split"
}

missing_cohort_columns = (
    required_cohort_columns
    - set(
        master_cohort_df.columns
    )
)

print(
    "코호트 누락 컬럼:",
    missing_cohort_columns
)

assert not missing_cohort_columns

assert master_cohort_df[
    "sample_id"
].is_unique

assert master_cohort_df[
    "churn"
].isin([0, 1]).all()

print("마스터 코호트 논리 검증 통과")

코호트 누락 컬럼: set()
마스터 코호트 논리 검증 통과


In [6]:
# 4. 월별 활동 데이터 불러오기
restaurant_monthly_df = pd.read_parquet(
    RESTAURANT_MONTHLY_PATH
)

culinary_monthly_df = pd.read_parquet(
    CULINARY_MONTHLY_PATH
)

In [7]:
expanded_monthly_df = (
    pd.concat(
        [
            restaurant_monthly_df,
            culinary_monthly_df
        ],
        ignore_index=True
    )
    .groupby(
        [
            "user_id",
            "year",
            "month"
        ],
        as_index=False
    )
    .agg(
        review_count=(
            "review_count",
            "sum"
        )
    )
)

print(
    "월별 활동 데이터:",
    expanded_monthly_df.shape
)

print(
    "고유 사용자:",
    expanded_monthly_df[
        "user_id"
    ].nunique()
)

print(
    "연도 범위:",
    expanded_monthly_df["year"].min(),
    "~",
    expanded_monthly_df["year"].max()
)

월별 활동 데이터: (3116516, 4)
고유 사용자: 1479066
연도 범위: 2005 ~ 2022


In [8]:
assert expanded_monthly_df[
    [
        "user_id",
        "year",
        "month",
        "review_count"
    ]
].isna().sum().sum() == 0

assert (
    expanded_monthly_df[
        "review_count"
    ]
    > 0
).all()

assert expanded_monthly_df[
    "month"
].between(1, 12).all()

print("월별 활동 데이터 검증 통과")

월별 활동 데이터 검증 통과


In [9]:
# 5. 사용자별 연간 활동량 생성
yearly_activity_df = (
    expanded_monthly_df
    .groupby(
        [
            "user_id",
            "year"
        ],
        as_index=False
    )
    .agg(
        review_count=(
            "review_count",
            "sum"
        ),
        active_months=(
            "month",
            "nunique"
        )
    )
)

yearly_activity_df[
    "reviews_per_active_month"
] = (
    yearly_activity_df[
        "review_count"
    ]
    / yearly_activity_df[
        "active_months"
    ]
)

yearly_activity_df.head()

,user_id,year,review_count,active_months,reviews_per_active_month
0,---2PmXbF47D870stH1jqA,2012,3,2,1.500000
1,---2PmXbF47D870stH1jqA,2013,2,2,1.000000
2,---2PmXbF47D870stH1jqA,2014,8,5,1.600000
3,---2PmXbF47D870stH1jqA,2015,4,4,1.000000
4,---2PmXbF47D870stH1jqA,2016,4,3,1.333333


In [10]:
assert yearly_activity_df[
    [
        "user_id",
        "year",
        "review_count",
        "active_months",
        "reviews_per_active_month"
    ]
].isna().sum().sum() == 0

assert (
    yearly_activity_df[
        "review_count"
    ]
    > 0
).all()

assert yearly_activity_df[
    "active_months"
].between(1, 12).all()

print("연간 활동량 생성 통과")

연간 활동량 생성 통과


In [11]:
# 6. Baseline 활동 피처 생성
baseline_activity_df = (
    yearly_activity_df
    .rename(
        columns={
            "year":
                "selection_year",
            "review_count":
                "baseline_review_count",
            "active_months":
                "baseline_active_months",
            "reviews_per_active_month":
                "baseline_reviews_per_active_month"
        }
    )
)

baseline_activity_df.head()

,user_id,selection_year,baseline_review_count,baseline_active_months,baseline_reviews_per_active_month
0,---2PmXbF47D870stH1jqA,2012,3,2,1.500000
1,---2PmXbF47D870stH1jqA,2013,2,2,1.000000
2,---2PmXbF47D870stH1jqA,2014,8,5,1.600000
3,---2PmXbF47D870stH1jqA,2015,4,4,1.000000
4,---2PmXbF47D870stH1jqA,2016,4,3,1.333333


In [12]:
# 7. Recent 활동 피처 생성
recent_activity_df = (
    yearly_activity_df
    .copy()
)

recent_activity_df[
    "selection_year"
] = (
    recent_activity_df[
        "year"
    ]
    - 1
)

recent_activity_df = (
    recent_activity_df
    .drop(
        columns=[
            "year"
        ]
    )
    .rename(
        columns={
            "review_count":
                "recent_review_count",
            "active_months":
                "recent_active_months",
            "reviews_per_active_month":
                "recent_reviews_per_active_month"
        }
    )
)

recent_activity_df.head()

,user_id,recent_review_count,recent_active_months,recent_reviews_per_active_month,selection_year
0,---2PmXbF47D870stH1jqA,3,2,1.500000,2011
1,---2PmXbF47D870stH1jqA,2,2,1.000000,2012
2,---2PmXbF47D870stH1jqA,8,5,1.600000,2013
3,---2PmXbF47D870stH1jqA,4,4,1.000000,2014
4,---2PmXbF47D870stH1jqA,4,3,1.333333,2015


In [13]:
# 8. 마스터 코호트와 결합
activity_feature_df = (
    master_cohort_df[
        [
            "sample_id",
            "user_id",
            "selection_year"
        ]
    ]
    .merge(
        baseline_activity_df,
        on=[
            "user_id",
            "selection_year"
        ],
        how="left",
        validate="one_to_one"
    )
    .merge(
        recent_activity_df,
        on=[
            "user_id",
            "selection_year"
        ],
        how="left",
        validate="one_to_one"
    )
)

print(
    "활동 피처 데이터:",
    activity_feature_df.shape
)

activity_feature_df.head()

활동 피처 데이터: (21601, 9)


,sample_id,user_id,selection_year,baseline_review_count,baseline_active_months,baseline_reviews_per_active_month,recent_review_count,recent_active_months,recent_reviews_per_active_month
0,-KICU2HksIrtaOymb_jqPQ_2009,-KICU2HksIrtaOymb_jqPQ,2009,23,8,2.875000,2,2,1.000000
1,-VuSUcCZCbQcdSdF7w9USg_2009,-VuSUcCZCbQcdSdF7w9USg,2009,11,4,2.750000,3,1,3.000000
2,-hKniZN2OdshWLHYuj21jQ_2009,-hKniZN2OdshWLHYuj21jQ,2009,12,6,2.000000,19,7,2.714286
3,-ju8d9NY3yZyVJCdw5oIUw_2009,-ju8d9NY3yZyVJCdw5oIUw,2009,11,4,2.750000,6,4,1.500000
4,-y-R9jOTso_XAjDOrabdFg_2009,-y-R9jOTso_XAjDOrabdFg,2009,22,7,3.142857,17,8,2.125000


In [14]:
# 9. 활동 변화 피처 생성
activity_feature_df[
    "review_count_diff"
] = (
    activity_feature_df[
        "recent_review_count"
    ]
    - activity_feature_df[
        "baseline_review_count"
    ]
)

activity_feature_df[
    "review_count_ratio"
] = (
    activity_feature_df[
        "recent_review_count"
    ]
    / activity_feature_df[
        "baseline_review_count"
    ]
)

activity_feature_df[
    "review_count_decline_rate"
] = (
    (
        activity_feature_df[
            "baseline_review_count"
        ]
        - activity_feature_df[
            "recent_review_count"
        ]
    )
    / activity_feature_df[
        "baseline_review_count"
    ]
)

In [15]:
# 활동 월수 변화
activity_feature_df[
    "active_month_diff"
] = (
    activity_feature_df[
        "recent_active_months"
    ]
    - activity_feature_df[
        "baseline_active_months"
    ]
)

activity_feature_df[
    "active_month_ratio"
] = (
    activity_feature_df[
        "recent_active_months"
    ]
    / activity_feature_df[
        "baseline_active_months"
    ]
)

activity_feature_df[
    "active_month_decline_rate"
] = (
    (
        activity_feature_df[
            "baseline_active_months"
        ]
        - activity_feature_df[
            "recent_active_months"
        ]
    )
    / activity_feature_df[
        "baseline_active_months"
    ]
)

In [16]:
# 활동 월당 리뷰 수 변화
activity_feature_df[
    "reviews_per_active_month_diff"
] = (
    activity_feature_df[
        "recent_reviews_per_active_month"
    ]
    - activity_feature_df[
        "baseline_reviews_per_active_month"
    ]
)

activity_feature_df[
    "reviews_per_active_month_ratio"
] = (
    activity_feature_df[
        "recent_reviews_per_active_month"
    ]
    / activity_feature_df[
        "baseline_reviews_per_active_month"
    ]
)

activity_feature_df[
    "reviews_per_active_month_decline_rate"
] = (
    (
        activity_feature_df[
            "baseline_reviews_per_active_month"
        ]
        - activity_feature_df[
            "recent_reviews_per_active_month"
        ]
    )
    / activity_feature_df[
        "baseline_reviews_per_active_month"
    ]
)

In [17]:
# 10. 활동 피처 품질 검증
feature_columns = [
    column
    for column in activity_feature_df.columns
    if column
    not in {
        "sample_id",
        "user_id",
        "selection_year"
    }
]

activity_feature_validation_df = pd.DataFrame(
    {
        "feature": feature_columns,
        "missing_count": [
            activity_feature_df[
                column
            ].isna().sum()
            for column in feature_columns
        ],
        "infinite_count": [
            np.isinf(
                activity_feature_df[
                    column
                ].astype(float)
            ).sum()
            for column in feature_columns
        ]
    }
)

activity_feature_validation_df

,feature,missing_count,infinite_count
0,baseline_review_count,0,0
1,baseline_active_months,0,0
2,baseline_reviews_per_active_month,0,0
3,recent_review_count,0,0
4,recent_active_months,0,0
5,recent_reviews_per_active_month,0,0
6,review_count_diff,0,0
7,review_count_ratio,0,0
8,review_count_decline_rate,0,0
9,active_month_diff,0,0


In [18]:
assert len(
    activity_feature_df
) == len(
    master_cohort_df
)

assert activity_feature_df[
    "sample_id"
].is_unique

assert activity_feature_df[
    [
        "sample_id",
        "user_id",
        "selection_year"
    ]
].isna().sum().sum() == 0

assert (
    activity_feature_validation_df[
        "missing_count"
    ].sum()
    == 0
)

assert (
    activity_feature_validation_df[
        "infinite_count"
    ].sum()
    == 0
)

assert (
    activity_feature_df[
        "baseline_review_count"
    ]
    >= 10
).all()

assert (
    activity_feature_df[
        "baseline_active_months"
    ]
    >= 3
).all()

assert (
    activity_feature_df[
        "recent_review_count"
    ]
    >= 1
).all()

print("롤링 활동 피처 논리 검증 통과")

롤링 활동 피처 논리 검증 통과


In [19]:
# 11. 기존 코호트 값과 재계산 결과 비교
if {
    "baseline_review_count",
    "baseline_active_months"
}.issubset(
    master_cohort_df.columns
):
    recalculation_validation_df = (
        master_cohort_df[
            [
                "sample_id",
                "baseline_review_count",
                "baseline_active_months"
            ]
        ]
        .merge(
            activity_feature_df[
                [
                    "sample_id",
                    "baseline_review_count",
                    "baseline_active_months"
                ]
            ],
            on="sample_id",
            how="left",
            suffixes=(
                "_cohort",
                "_recalculated"
            ),
            validate="one_to_one"
        )
    )

    assert (
        recalculation_validation_df[
            "baseline_review_count_cohort"
        ]
        == recalculation_validation_df[
            "baseline_review_count_recalculated"
        ]
    ).all()

    assert (
        recalculation_validation_df[
            "baseline_active_months_cohort"
        ]
        == recalculation_validation_df[
            "baseline_active_months_recalculated"
        ]
    ).all()

    print("코호트 활동량 재계산 검증 통과")

코호트 활동량 재계산 검증 통과


In [20]:
# 12. 이탈·유지 집단 비교
activity_analysis_df = (
    activity_feature_df
    .merge(
        master_cohort_df[
            [
                "sample_id",
                "churn"
            ]
        ],
        on="sample_id",
        how="left",
        validate="one_to_one"
    )
)

activity_churn_summary_df = (
    activity_analysis_df
    .groupby(
        "churn",
        as_index=False
    )
    .agg(
        users=(
            "sample_id",
            "size"
        ),
        baseline_review_count_mean=(
            "baseline_review_count",
            "mean"
        ),
        recent_review_count_mean=(
            "recent_review_count",
            "mean"
        ),
        review_count_decline_rate_mean=(
            "review_count_decline_rate",
            "mean"
        ),
        baseline_active_months_mean=(
            "baseline_active_months",
            "mean"
        ),
        recent_active_months_mean=(
            "recent_active_months",
            "mean"
        ),
        active_month_decline_rate_mean=(
            "active_month_decline_rate",
            "mean"
        ),
        reviews_per_active_month_decline_rate_mean=(
            "reviews_per_active_month_decline_rate",
            "mean"
        )
    )
)

activity_churn_summary_df

,churn,users,baseline_review_count_mean,recent_review_count_mean,review_count_decline_rate_mean,baseline_active_months_mean,recent_active_months_mean,active_month_decline_rate_mean,reviews_per_active_month_decline_rate_mean
0,0,18215,25.061378,19.022234,0.165517,7.008674,6.391930,0.024213,0.171115
1,1,3386,19.671294,8.118133,0.554100,5.753692,3.580626,0.333684,0.339885


In [21]:
# 13. 저장
activity_feature_df.to_parquet(
    ACTIVITY_FEATURE_PATH,
    index=False
)

activity_feature_validation_df.to_csv(
    REPORT_TABLE_DIR
    / "rolling_activity_feature_validation_v02.csv",
    index=False,
    encoding="utf-8-sig"
)

activity_churn_summary_df.to_csv(
    REPORT_TABLE_DIR
    / "rolling_activity_churn_summary_v02.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "활동 피처 저장:",
    ACTIVITY_FEATURE_PATH
)

활동 피처 저장: C:\Users\playdata2\SKN34-2nd-5Team\data\interim\features_rolling\activity_features_rolling_v02.parquet


In [22]:
year_cohort_summary_df = (
    master_cohort_df
    .groupby(
        "selection_year",
        as_index=False
    )
    .agg(
        samples=(
            "sample_id",
            "size"
        ),
        unique_users=(
            "user_id",
            "nunique"
        ),
        churn_users=(
            "churn",
            "sum"
        )
    )
)

year_cohort_summary_df[
    "churn_rate_pct"
] = (
    year_cohort_summary_df[
        "churn_users"
    ]
    / year_cohort_summary_df[
        "samples"
    ]
    * 100
).round(2)

year_cohort_summary_df

,selection_year,samples,unique_users,churn_users,churn_rate_pct
0,2009,550,550,78,14.18
1,2010,996,996,180,18.07
2,2011,1586,1586,275,17.34
3,2012,1834,1834,275,14.99
4,2013,2324,2324,363,15.62
5,2014,2917,2917,461,15.80
6,2015,3513,3513,557,15.86
7,2016,3724,3724,527,14.15
8,2017,4157,4157,670,16.12


In [23]:
print(
    "최종 활동 피처 크기:",
    activity_feature_df.shape
)

print(
    "최종 피처 수:",
    len(
        activity_feature_df.columns
    )
)

최종 활동 피처 크기: (21601, 18)
최종 피처 수: 18


In [24]:
print(
    "모델 후보 피처 수:",
    len(feature_columns)
)

모델 후보 피처 수: 15


In [25]:
candidate_split_summary_df = (
    master_cohort_df[
        master_cohort_df[
            "candidate_range_split"
        ]
        != "excluded"
    ]
    .groupby(
        "candidate_range_split",
        as_index=False
    )
    .agg(
        samples=(
            "sample_id",
            "size"
        ),
        unique_users=(
            "user_id",
            "nunique"
        ),
        churn_users=(
            "churn",
            "sum"
        )
    )
)

candidate_split_summary_df[
    "churn_rate_pct"
] = (
    candidate_split_summary_df[
        "churn_users"
    ]
    / candidate_split_summary_df[
        "samples"
    ]
    * 100
).round(2)

candidate_split_summary_df

,candidate_range_split,samples,unique_users,churn_users,churn_rate_pct
0,test,4157,4157,670,16.12
1,train,13720,8483,2189,15.95
2,validation,3724,3724,527,14.15


In [26]:
recent_split_summary_df = (
    master_cohort_df[
        master_cohort_df[
            "recent_range_split"
        ]
        != "excluded"
    ]
    .groupby(
        "recent_range_split",
        as_index=False
    )
    .agg(
        samples=(
            "sample_id",
            "size"
        ),
        unique_users=(
            "user_id",
            "nunique"
        ),
        churn_users=(
            "churn",
            "sum"
        )
    )
)

recent_split_summary_df[
    "churn_rate_pct"
] = (
    recent_split_summary_df[
        "churn_users"
    ]
    / recent_split_summary_df[
        "samples"
    ]
    * 100
).round(2)

recent_split_summary_df

,recent_range_split,samples,unique_users,churn_users,churn_rate_pct
0,test,4157,4157,670,16.12
1,train,8754,6211,1381,15.78
2,validation,3724,3724,527,14.15


In [27]:
# 14. 리뷰 상세 데이터 준비
# 14-1. 원본·캐시 경로 설정
RAW_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
)

BUSINESS_JSON_PATH = (
    RAW_DIR
    / "yelp_academic_dataset_business.json"
)

REVIEW_JSON_PATH = (
    RAW_DIR
    / "yelp_academic_dataset_review.json"
)

RESTAURANT_REVIEW_PATH = (
    INTERIM_DIR
    / "restaurant_reviews.parquet"
)

ADDITIONAL_CULINARY_BUSINESS_PATH = (
    INTERIM_DIR
    / "additional_culinary_businesses_v02.parquet"
)

ADDITIONAL_CULINARY_REVIEW_PATH = (
    INTERIM_DIR
    / "additional_culinary_reviews_v02.parquet"
)

INTERVAL_FEATURE_PATH = (
    ROLLING_FEATURE_DIR
    / "interval_features_rolling_v02.parquet"
)

print(
    "Business JSON:",
    BUSINESS_JSON_PATH.exists()
)

print(
    "Review JSON:",
    REVIEW_JSON_PATH.exists()
)

print(
    "Restaurants 리뷰:",
    RESTAURANT_REVIEW_PATH.exists()
)

assert BUSINESS_JSON_PATH.exists()
assert REVIEW_JSON_PATH.exists()
assert RESTAURANT_REVIEW_PATH.exists()

Business JSON: True
Review JSON: True
Restaurants 리뷰: True


In [28]:
# 14-2. 미식 방문형 카테고리 정의
CULINARY_VISIT_CATEGORIES = {
    "Cafes",
    "Coffee & Tea",
    "Ice Cream & Frozen Yogurt",
    "Desserts",
    "Bakeries",
    "Juice Bars & Smoothies",
    "Donuts",
    "Cupcakes",
    "Food Trucks",
    "Bubble Tea",
    "Shaved Ice"
}

In [29]:
# 14-3. 추가 미식 방문형 업체 생성
if ADDITIONAL_CULINARY_BUSINESS_PATH.exists():
    additional_culinary_business_df = (
        pd.read_parquet(
            ADDITIONAL_CULINARY_BUSINESS_PATH
        )
    )

    print(
        "저장된 미식 방문형 업체를 불러왔습니다."
    )

else:
    business_full_df = pd.read_json(
        BUSINESS_JSON_PATH,
        lines=True
    )

    business_full_df[
        "category_list"
    ] = (
        business_full_df[
            "categories"
        ]
        .fillna("")
        .str.split(", ")
    )

    business_full_df[
        "is_restaurant"
    ] = (
        business_full_df[
            "category_list"
        ]
        .apply(
            lambda categories:
                "Restaurants"
                in categories
        )
    )

    business_full_df[
        "is_food"
    ] = (
        business_full_df[
            "category_list"
        ]
        .apply(
            lambda categories:
                "Food"
                in categories
        )
    )

    business_full_df[
        "is_culinary_visit"
    ] = (
        business_full_df[
            "category_list"
        ]
        .apply(
            lambda categories:
                bool(
                    set(categories)
                    & CULINARY_VISIT_CATEGORIES
                )
        )
    )

    additional_culinary_business_df = (
        business_full_df[
            (
                ~business_full_df[
                    "is_restaurant"
                ]
            )
            & (
                business_full_df[
                    "is_food"
                ]
            )
            & (
                business_full_df[
                    "is_culinary_visit"
                ]
            )
        ]
        [
            [
                "business_id",
                "name",
                "address",
                "city",
                "state",
                "postal_code",
                "latitude",
                "longitude",
                "stars",
                "review_count",
                "is_open",
                "categories"
            ]
        ]
        .copy()
        .reset_index(drop=True)
    )

    additional_culinary_business_df.to_parquet(
        ADDITIONAL_CULINARY_BUSINESS_PATH,
        index=False
    )

    print(
        "미식 방문형 업체 파일 저장 완료"
    )

미식 방문형 업체 파일 저장 완료


In [30]:
print(
    "추가 미식 방문형 업체:",
    len(
        additional_culinary_business_df
    )
)

print(
    "고유 업체:",
    additional_culinary_business_df[
        "business_id"
    ].nunique()
)

assert (
    len(
        additional_culinary_business_df
    )
    == 5_888
)

assert additional_culinary_business_df[
    "business_id"
].is_unique

print("미식 방문형 업체 검증 통과")

추가 미식 방문형 업체: 5888
고유 업체: 5888
미식 방문형 업체 검증 통과


In [31]:
# 14-4. 추가 미식 방문형 Review 추출

# 추가 리뷰는 225,793건이므로 필요한 컬럼만 저장한다.

additional_culinary_business_ids = set(
    additional_culinary_business_df[
        "business_id"
    ]
)

REVIEW_DETAIL_COLUMNS = [
    "review_id",
    "user_id",
    "business_id",
    "stars",
    "useful",
    "funny",
    "cool",
    "date"
]

In [32]:
if ADDITIONAL_CULINARY_REVIEW_PATH.exists():
    additional_culinary_review_df = (
        pd.read_parquet(
            ADDITIONAL_CULINARY_REVIEW_PATH
        )
    )

    print(
        "저장된 미식 방문형 리뷰를 불러왔습니다."
    )

else:
    culinary_review_parts = []

    for chunk_number, chunk_df in enumerate(
        pd.read_json(
            REVIEW_JSON_PATH,
            lines=True,
            chunksize=250_000
        ),
        start=1
    ):
        selected_chunk_df = (
            chunk_df[
                chunk_df[
                    "business_id"
                ].isin(
                    additional_culinary_business_ids
                )
            ]
            [
                REVIEW_DETAIL_COLUMNS
            ]
            .copy()
        )

        if selected_chunk_df.empty:
            continue

        culinary_review_parts.append(
            selected_chunk_df
        )

        print(
            f"{chunk_number}번째 청크: "
            f"{len(selected_chunk_df):,}건"
        )

    additional_culinary_review_df = (
        pd.concat(
            culinary_review_parts,
            ignore_index=True
        )
    )

    additional_culinary_review_df[
        "date"
    ] = pd.to_datetime(
        additional_culinary_review_df[
            "date"
        ],
        errors="coerce"
    )

    additional_culinary_review_df.to_parquet(
        ADDITIONAL_CULINARY_REVIEW_PATH,
        index=False
    )

    print(
        "미식 방문형 리뷰 파일 저장 완료"
    )

1번째 청크: 8,131건
2번째 청크: 7,986건
3번째 청크: 8,393건
4번째 청크: 7,414건
5번째 청크: 7,975건
6번째 청크: 7,931건
7번째 청크: 6,398건
8번째 청크: 7,150건
9번째 청크: 8,042건
10번째 청크: 7,890건
11번째 청크: 8,244건
12번째 청크: 9,333건
13번째 청크: 9,535건
14번째 청크: 10,139건
15번째 청크: 8,061건
16번째 청크: 7,714건
17번째 청크: 7,859건
18번째 청크: 7,094건
19번째 청크: 8,068건
20번째 청크: 8,867건
21번째 청크: 8,252건
22번째 청크: 8,604건
23번째 청크: 7,739건
24번째 청크: 6,875건
25번째 청크: 7,446건
26번째 청크: 8,552건
27번째 청크: 8,157건
28번째 청크: 7,944건
미식 방문형 리뷰 파일 저장 완료


In [33]:
print(
    "추가 미식 방문형 리뷰:",
    len(
        additional_culinary_review_df
    )
)

print(
    "고유 리뷰 ID:",
    additional_culinary_review_df[
        "review_id"
    ].nunique()
)

print(
    "고유 사용자:",
    additional_culinary_review_df[
        "user_id"
    ].nunique()
)

print(
    "날짜 범위:",
    additional_culinary_review_df[
        "date"
    ].min(),
    "~",
    additional_culinary_review_df[
        "date"
    ].max()
)

assert (
    len(
        additional_culinary_review_df
    )
    == 225_793
)

assert additional_culinary_review_df[
    "review_id"
].is_unique

assert additional_culinary_review_df[
    [
        "review_id",
        "user_id",
        "business_id",
        "date"
    ]
].isna().sum().sum() == 0

print("미식 방문형 리뷰 검증 통과")

추가 미식 방문형 리뷰: 225793
고유 리뷰 ID: 225793
고유 사용자: 140133
날짜 범위: 2005-03-12 18:18:34 ~ 2022-01-19 19:45:44
미식 방문형 리뷰 검증 통과


In [34]:
# 14-5. 작성 날짜 데이터 결합

# 작성 간격 계산에는 user_id, date만 있으면 돼.
restaurant_review_date_df = (
    pd.read_parquet(
        RESTAURANT_REVIEW_PATH,
        columns=[
            "user_id",
            "date"
        ]
    )
)

restaurant_review_date_df[
    "date"
] = pd.to_datetime(
    restaurant_review_date_df[
        "date"
    ],
    errors="coerce"
)

additional_review_date_df = (
    additional_culinary_review_df[
        [
            "user_id",
            "date"
        ]
    ]
    .copy()
)

review_date_df = (
    pd.concat(
        [
            restaurant_review_date_df,
            additional_review_date_df
        ],
        ignore_index=True
    )
    .dropna(
        subset=[
            "user_id",
            "date"
        ]
    )
)

print(
    "결합 리뷰 날짜:",
    review_date_df.shape
)

print(
    "전체 리뷰 수:",
    f"{len(review_date_df):,}"
)

assert (
    len(review_date_df)
    == 4_950_264
)

print("리뷰 상세 날짜 결합 통과")

결합 리뷰 날짜: (4950264, 2)
전체 리뷰 수: 4,950,264
리뷰 상세 날짜 결합 통과


In [35]:
# 14-6. 코호트 사용자와 필요 연도만 필터링

# 전체 리뷰를 계속 들고 있지 않고 피처 생성에 필요한 사용자·연도만 남긴다.
cohort_user_ids = set(
    master_cohort_df[
        "user_id"
    ]
)

minimum_feature_year = int(
    master_cohort_df[
        "selection_year"
    ].min()
)

maximum_feature_year = int(
    master_cohort_df[
        "observation_year"
    ].max()
)

review_date_df[
    "review_year"
] = (
    review_date_df[
        "date"
    ]
    .dt.year
    .astype("int16")
)

cohort_review_date_df = (
    review_date_df[
        (
            review_date_df[
                "user_id"
            ].isin(
                cohort_user_ids
            )
        )
        & (
            review_date_df[
                "review_year"
            ].between(
                minimum_feature_year,
                maximum_feature_year
            )
        )
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "피처 생성 대상 리뷰:",
    cohort_review_date_df.shape
)

print(
    "대상 사용자:",
    cohort_review_date_df[
        "user_id"
    ].nunique()
)

print(
    "대상 연도:",
    cohort_review_date_df[
        "review_year"
    ].min(),
    "~",
    cohort_review_date_df[
        "review_year"
    ].max()
)

피처 생성 대상 리뷰: (771953, 3)
대상 사용자: 12709
대상 연도: 2009 ~ 2018


In [36]:
del restaurant_review_date_df
del additional_review_date_df
del review_date_df

In [37]:
# 15. 작성 간격 피처 생성
# 15-1. Baseline 리뷰 연결
baseline_review_period_df = (
    cohort_review_date_df[
        [
            "user_id",
            "date",
            "review_year"
        ]
    ]
    .rename(
        columns={
            "review_year":
                "selection_year"
        }
    )
)

baseline_review_period_df[
    "period"
] = "baseline"

In [38]:
# 15-2. Recent 리뷰 연결
recent_review_period_df = (
    cohort_review_date_df[
        [
            "user_id",
            "date",
            "review_year"
        ]
    ]
    .copy()
)

recent_review_period_df[
    "selection_year"
] = (
    recent_review_period_df[
        "review_year"
    ]
    - 1
)

recent_review_period_df = (
    recent_review_period_df
    .drop(
        columns=[
            "review_year"
        ]
    )
)

recent_review_period_df[
    "period"
] = "recent"

In [39]:
# 15-3. 마스터 코호트와 연결
period_review_df = (
    pd.concat(
        [
            baseline_review_period_df,
            recent_review_period_df
        ],
        ignore_index=True
    )
    .merge(
        master_cohort_df[
            [
                "sample_id",
                "user_id",
                "selection_year"
            ]
        ],
        on=[
            "user_id",
            "selection_year"
        ],
        how="inner",
        validate="many_to_one"
    )
)

period_review_df = (
    period_review_df
    .sort_values(
        [
            "sample_id",
            "period",
            "date"
        ]
    )
    .reset_index(drop=True)
)

print(
    "코호트 기간 리뷰:",
    period_review_df.shape
)

print(
    period_review_df[
        "period"
    ].value_counts()
)

코호트 기간 리뷰: (897078, 5)
period
baseline    523100
recent      373978
Name: count, dtype: int64


In [41]:
# Baseline·Recent 시간 범위 검증

baseline_validation_df = (
    period_review_df.loc[
        period_review_df[
            "period"
        ]
        == "baseline",
        [
            "sample_id",
            "date",
            "selection_year"
        ]
    ]
    .copy()
)

baseline_validation_df[
    "review_year"
] = (
    baseline_validation_df[
        "date"
    ]
    .dt.year
)

baseline_validation_df[
    "year_match"
] = (
    baseline_validation_df[
        "review_year"
    ]
    == baseline_validation_df[
        "selection_year"
    ]
)


recent_validation_df = (
    period_review_df.loc[
        period_review_df[
            "period"
        ]
        == "recent",
        [
            "sample_id",
            "date",
            "selection_year"
        ]
    ]
    .copy()
)

recent_validation_df[
    "review_year"
] = (
    recent_validation_df[
        "date"
    ]
    .dt.year
)

recent_validation_df[
    "expected_review_year"
] = (
    recent_validation_df[
        "selection_year"
    ]
    + 1
)

recent_validation_df[
    "year_match"
] = (
    recent_validation_df[
        "review_year"
    ]
    == recent_validation_df[
        "expected_review_year"
    ]
)

In [42]:
baseline_mismatch_count = int(
    (
        ~baseline_validation_df[
            "year_match"
        ]
    ).sum()
)

recent_mismatch_count = int(
    (
        ~recent_validation_df[
            "year_match"
        ]
    ).sum()
)

print(
    "Baseline 연도 불일치:",
    baseline_mismatch_count
)

print(
    "Recent 연도 불일치:",
    recent_mismatch_count
)

Baseline 연도 불일치: 0
Recent 연도 불일치: 0


In [43]:
assert baseline_mismatch_count == 0
assert recent_mismatch_count == 0

print("Baseline·Recent 시간 범위 검증 통과")

Baseline·Recent 시간 범위 검증 통과


In [ ]:
# 15-4. 리뷰 작성 간격 계산
period_review_df[
    "interval_days"
] = (
    period_review_df
    .groupby(
        [
            "sample_id",
            "period"
        ]
    )
    [
        "date"
    ]
    .diff()
    .dt.total_seconds()
    / 86_400
)

In [45]:
interval_period_summary_df = (
    period_review_df
    .groupby(
        [
            "sample_id",
            "period"
        ],
        as_index=False
    )
    .agg(
        review_count=(
            "date",
            "size"
        ),
        mean_interval_days=(
            "interval_days",
            "mean"
        ),
        median_interval_days=(
            "interval_days",
            "median"
        ),
        max_interval_days=(
            "interval_days",
            "max"
        ),
        last_review_date=(
            "date",
            "max"
        )
    )
)

interval_period_summary_df.head()

,sample_id,period,review_count,mean_interval_days,median_interval_days,max_interval_days,last_review_date
0,--Vu3Gux9nPnLcG9yO_HxA_2017,baseline,22,6.963668,0.593125,57.164387,2017-08-13 20:37:44
1,--Vu3Gux9nPnLcG9yO_HxA_2017,recent,4,69.701578,82.129896,108.570220,2018-09-20 18:21:38
2,--u09WAjW741FdfkJXxNmg_2017,baseline,19,17.434153,18.273872,41.611748,2017-11-18 19:14:59
3,--u09WAjW741FdfkJXxNmg_2017,recent,12,32.103809,29.163137,57.695139,2018-12-23 01:52:34
4,-00kdEIhCt-ODaV4BS-EAg_2012,baseline,12,32.177315,7.418009,134.852882,2012-12-27 23:07:33


In [46]:
# 15-5. 연말 기준 최근 활동 공백 계산
interval_period_summary_df = (
    interval_period_summary_df
    .merge(
        master_cohort_df[
            [
                "sample_id",
                "selection_year"
            ]
        ],
        on="sample_id",
        how="left",
        validate="many_to_one"
    )
)

interval_period_summary_df[
    "period_end_year"
] = np.where(
    interval_period_summary_df[
        "period"
    ]
    == "baseline",
    interval_period_summary_df[
        "selection_year"
    ]
    + 1,
    interval_period_summary_df[
        "selection_year"
    ]
    + 2
)

interval_period_summary_df[
    "period_end_date"
] = pd.to_datetime(
    interval_period_summary_df[
        "period_end_year"
    ].astype(str)
    + "-01-01"
)

interval_period_summary_df[
    "recency_days"
] = (
    interval_period_summary_df[
        "period_end_date"
    ]
    - interval_period_summary_df[
        "last_review_date"
    ]
).dt.total_seconds() / 86_400

assert (
    interval_period_summary_df[
        "recency_days"
    ]
    >= 0
).all()

interval_period_summary_df.head()

,sample_id,period,review_count,mean_interval_days,median_interval_days,max_interval_days,last_review_date,selection_year,period_end_year,period_end_date,recency_days
0,--Vu3Gux9nPnLcG9yO_HxA_2017,baseline,22,6.963668,0.593125,57.164387,2017-08-13 20:37:44,2017,2018,2018-01-01,140.140463
1,--Vu3Gux9nPnLcG9yO_HxA_2017,recent,4,69.701578,82.129896,108.570220,2018-09-20 18:21:38,2017,2019,2019-01-01,102.234977
2,--u09WAjW741FdfkJXxNmg_2017,baseline,19,17.434153,18.273872,41.611748,2017-11-18 19:14:59,2017,2018,2018-01-01,43.197928
3,--u09WAjW741FdfkJXxNmg_2017,recent,12,32.103809,29.163137,57.695139,2018-12-23 01:52:34,2017,2019,2019-01-01,8.921829
4,-00kdEIhCt-ODaV4BS-EAg_2012,baseline,12,32.177315,7.418009,134.852882,2012-12-27 23:07:33,2012,2013,2013-01-01,4.036424


In [47]:
# 15-6. Baseline·Recent 피처 분리
baseline_interval_df = (
    interval_period_summary_df[
        interval_period_summary_df[
            "period"
        ]
        == "baseline"
    ]
    [
        [
            "sample_id",
            "mean_interval_days",
            "median_interval_days",
            "max_interval_days",
            "recency_days"
        ]
    ]
    .rename(
        columns={
            "mean_interval_days":
                "baseline_mean_interval_days",
            "median_interval_days":
                "baseline_median_interval_days",
            "max_interval_days":
                "baseline_max_interval_days",
            "recency_days":
                "baseline_recency_days"
        }
    )
)

recent_interval_df = (
    interval_period_summary_df[
        interval_period_summary_df[
            "period"
        ]
        == "recent"
    ]
    [
        [
            "sample_id",
            "mean_interval_days",
            "median_interval_days",
            "max_interval_days",
            "recency_days"
        ]
    ]
    .rename(
        columns={
            "mean_interval_days":
                "recent_mean_interval_days",
            "median_interval_days":
                "recent_median_interval_days",
            "max_interval_days":
                "recent_max_interval_days",
            "recency_days":
                "recent_recency_days"
        }
    )
)

In [48]:
# 15-7. 간격 피처 결합
interval_feature_df = (
    master_cohort_df[
        [
            "sample_id",
            "user_id",
            "selection_year"
        ]
    ]
    .merge(
        baseline_interval_df,
        on="sample_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        recent_interval_df,
        on="sample_id",
        how="left",
        validate="one_to_one"
    )
)

In [49]:
activity_count_df = (
    activity_feature_df[
        [
            "sample_id",
            "baseline_review_count",
            "recent_review_count"
        ]
    ]
)

interval_feature_df = (
    interval_feature_df
    .merge(
        activity_count_df,
        on="sample_id",
        how="left",
        validate="one_to_one"
    )
)

interval_feature_df[
    "recent_interval_available"
] = (
    interval_feature_df[
        "recent_review_count"
    ]
    >= 2
).astype("int8")

In [50]:
# 15-8. 증가 피처 생성
interval_feature_df[
    "mean_interval_increase_days"
] = (
    interval_feature_df[
        "recent_mean_interval_days"
    ]
    - interval_feature_df[
        "baseline_mean_interval_days"
    ]
)

interval_feature_df[
    "median_interval_increase_days"
] = (
    interval_feature_df[
        "recent_median_interval_days"
    ]
    - interval_feature_df[
        "baseline_median_interval_days"
    ]
)

interval_feature_df[
    "max_interval_increase_days"
] = (
    interval_feature_df[
        "recent_max_interval_days"
    ]
    - interval_feature_df[
        "baseline_max_interval_days"
    ]
)

interval_feature_df[
    "recency_increase_days"
] = (
    interval_feature_df[
        "recent_recency_days"
    ]
    - interval_feature_df[
        "baseline_recency_days"
    ]
)

In [51]:
interval_feature_df = (
    interval_feature_df
    .drop(
        columns=[
            "baseline_review_count",
            "recent_review_count"
        ]
    )
)

In [52]:
# 15-9. 간격 피처 품질 검증
interval_feature_columns = [
    column
    for column in interval_feature_df.columns
    if column
    not in {
        "sample_id",
        "user_id",
        "selection_year"
    }
]

interval_feature_validation_df = pd.DataFrame(
    {
        "feature":
            interval_feature_columns,
        "missing_count": [
            interval_feature_df[
                column
            ].isna().sum()
            for column
            in interval_feature_columns
        ],
        "infinite_count": [
            np.isinf(
                interval_feature_df[
                    column
                ]
                .fillna(0)
                .astype(float)
            ).sum()
            for column
            in interval_feature_columns
        ]
    }
)

interval_feature_validation_df

,feature,missing_count,infinite_count
0,baseline_mean_interval_days,0,0
1,baseline_median_interval_days,0,0
2,baseline_max_interval_days,0,0
3,baseline_recency_days,0,0
4,recent_mean_interval_days,858,0
5,recent_median_interval_days,858,0
6,recent_max_interval_days,858,0
7,recent_recency_days,0,0
8,recent_interval_available,0,0
9,mean_interval_increase_days,858,0


In [53]:
assert len(
    interval_feature_df
) == len(
    master_cohort_df
)

assert interval_feature_df[
    "sample_id"
].is_unique

assert interval_feature_df[
    [
        "sample_id",
        "user_id",
        "selection_year",
        "baseline_recency_days",
        "recent_recency_days",
        "recency_increase_days",
        "recent_interval_available"
    ]
].isna().sum().sum() == 0

assert (
    interval_feature_df[
        "recent_interval_available"
    ]
    .isin([0, 1])
    .all()
)

assert (
    interval_feature_validation_df[
        "infinite_count"
    ].sum()
    == 0
)

print("롤링 작성 간격 피처 논리 검증 통과")

롤링 작성 간격 피처 논리 검증 통과


In [54]:
recent_interval_unavailable_count = int(
    (
        interval_feature_df[
            "recent_interval_available"
        ]
        == 0
    ).sum()
)

expected_missing_columns = [
    "recent_mean_interval_days",
    "recent_median_interval_days",
    "recent_max_interval_days",
    "mean_interval_increase_days",
    "median_interval_increase_days",
    "max_interval_increase_days"
]

for column in expected_missing_columns:
    assert (
        interval_feature_df[
            column
        ].isna().sum()
        == recent_interval_unavailable_count
    )

print(
    "Recent 간격 계산 불가 사용자:",
    recent_interval_unavailable_count
)

print(
    "간격 결측 구조 검증 통과"
)

Recent 간격 계산 불가 사용자: 858
간격 결측 구조 검증 통과


In [55]:
# 15-10. 이탈·유지 집단 비교
interval_analysis_df = (
    interval_feature_df
    .merge(
        master_cohort_df[
            [
                "sample_id",
                "churn"
            ]
        ],
        on="sample_id",
        how="left",
        validate="one_to_one"
    )
)

In [56]:
interval_churn_summary_df = (
    interval_analysis_df
    .groupby(
        "churn",
        as_index=False
    )
    .agg(
        samples=(
            "sample_id",
            "size"
        ),
        baseline_mean_interval_days_mean=(
            "baseline_mean_interval_days",
            "mean"
        ),
        recent_mean_interval_days_mean=(
            "recent_mean_interval_days",
            "mean"
        ),
        mean_interval_increase_days_mean=(
            "mean_interval_increase_days",
            "mean"
        ),
        baseline_recency_days_mean=(
            "baseline_recency_days",
            "mean"
        ),
        recent_recency_days_mean=(
            "recent_recency_days",
            "mean"
        ),
        recency_increase_days_mean=(
            "recency_increase_days",
            "mean"
        ),
        recent_interval_available_rate=(
            "recent_interval_available",
            "mean"
        )
    )
)

interval_churn_summary_df[
    "recent_interval_available_rate"
] = (
    interval_churn_summary_df[
        "recent_interval_available_rate"
    ]
    * 100
).round(2)

interval_churn_summary_df

,churn,samples,baseline_mean_interval_days_mean,recent_mean_interval_days_mean,mean_interval_increase_days_mean,baseline_recency_days_mean,recent_recency_days_mean,recency_increase_days_mean,recent_interval_available_rate
0,0,18215,15.338837,30.866401,15.586521,38.038233,48.174290,10.136057,97.61
1,1,3386,15.976992,48.183530,32.309640,59.328629,92.501257,33.172628,87.51


In [57]:
# 15-11. 저장
interval_feature_df.to_parquet(
    INTERVAL_FEATURE_PATH,
    index=False
)

interval_feature_validation_df.to_csv(
    REPORT_TABLE_DIR
    / "rolling_interval_feature_validation_v02.csv",
    index=False,
    encoding="utf-8-sig"
)

interval_churn_summary_df.to_csv(
    REPORT_TABLE_DIR
    / "rolling_interval_churn_summary_v02.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "작성 간격 피처 저장:",
    INTERVAL_FEATURE_PATH
)

작성 간격 피처 저장: C:\Users\playdata2\SKN34-2nd-5Team\data\interim\features_rolling\interval_features_rolling_v02.parquet


In [58]:
# 16. 고유·신규 음식점 피처
# 16-1. 경로 설정
BUSINESS_FEATURE_PATH = (
    ROLLING_FEATURE_DIR
    / "business_features_rolling_v02.parquet"
)

In [59]:
# 16-2. Restaurants·미식 방문형 리뷰 결합
restaurant_business_review_df = (
    pd.read_parquet(
        RESTAURANT_REVIEW_PATH,
        columns=[
            "user_id",
            "business_id",
            "date"
        ]
    )
)

restaurant_business_review_df[
    "date"
] = pd.to_datetime(
    restaurant_business_review_df[
        "date"
    ],
    errors="coerce"
)

additional_business_review_df = (
    pd.read_parquet(
        ADDITIONAL_CULINARY_REVIEW_PATH,
        columns=[
            "user_id",
            "business_id",
            "date"
        ]
    )
)

additional_business_review_df[
    "date"
] = pd.to_datetime(
    additional_business_review_df[
        "date"
    ],
    errors="coerce"
)

combined_business_review_df = (
    pd.concat(
        [
            restaurant_business_review_df,
            additional_business_review_df
        ],
        ignore_index=True
    )
    .dropna(
        subset=[
            "user_id",
            "business_id",
            "date"
        ]
    )
)

print(
    "결합 리뷰:",
    f"{len(combined_business_review_df):,}건"
)

assert (
    len(combined_business_review_df)
    == 4_950_264
)

print("음식점 피처용 리뷰 결합 통과")

결합 리뷰: 4,950,264건
음식점 피처용 리뷰 결합 통과


In [60]:
# 16-3. 코호트 사용자 전체 리뷰 이력
cohort_user_ids = set(
    master_cohort_df[
        "user_id"
    ]
)

cohort_user_review_history_df = (
    combined_business_review_df[
        combined_business_review_df[
            "user_id"
        ].isin(
            cohort_user_ids
        )
    ]
    .copy()
    .reset_index(drop=True)
)

cohort_user_review_history_df[
    "review_year"
] = (
    cohort_user_review_history_df[
        "date"
    ]
    .dt.year
    .astype("int16")
)

print(
    "코호트 사용자 전체 리뷰 이력:",
    cohort_user_review_history_df.shape
)

코호트 사용자 전체 리뷰 이력: (903036, 4)


In [61]:
user_business_first_review_df = (
    cohort_user_review_history_df
    .groupby(
        [
            "user_id",
            "business_id"
        ],
        as_index=False
    )
    .agg(
        first_review_date=(
            "date",
            "min"
        )
    )
)

user_business_first_review_df[
    "first_review_year"
] = (
    user_business_first_review_df[
        "first_review_date"
    ]
    .dt.year
    .astype("int16")
)

print(
    "사용자×음식점 조합:",
    user_business_first_review_df.shape
)

사용자×음식점 조합: (837302, 4)


In [62]:
# 16-4. 피처 대상 연도 필터링
minimum_feature_year = int(
    master_cohort_df[
        "selection_year"
    ].min()
)

maximum_feature_year = int(
    master_cohort_df[
        "observation_year"
    ].max()
)

feature_business_review_df = (
    cohort_user_review_history_df[
        cohort_user_review_history_df[
            "review_year"
        ].between(
            minimum_feature_year,
            maximum_feature_year
        )
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "피처 대상 리뷰:",
    feature_business_review_df.shape
)

피처 대상 리뷰: (771953, 4)


In [63]:
# 16-5. Baseline·Recent 기간 연결
baseline_business_review_df = (
    feature_business_review_df[
        [
            "user_id",
            "business_id",
            "date",
            "review_year"
        ]
    ]
    .rename(
        columns={
            "review_year":
                "selection_year"
        }
    )
)

baseline_business_review_df[
    "period"
] = "baseline"

In [64]:
recent_business_review_df = (
    feature_business_review_df[
        [
            "user_id",
            "business_id",
            "date",
            "review_year"
        ]
    ]
    .copy()
)

recent_business_review_df[
    "selection_year"
] = (
    recent_business_review_df[
        "review_year"
    ]
    - 1
)

recent_business_review_df = (
    recent_business_review_df
    .drop(
        columns=[
            "review_year"
        ]
    )
)

recent_business_review_df[
    "period"
] = "recent"

In [65]:
period_business_review_df = (
    pd.concat(
        [
            baseline_business_review_df,
            recent_business_review_df
        ],
        ignore_index=True
    )
    .merge(
        master_cohort_df[
            [
                "sample_id",
                "user_id",
                "selection_year"
            ]
        ],
        on=[
            "user_id",
            "selection_year"
        ],
        how="inner",
        validate="many_to_one"
    )
)

print(
    "코호트 기간 음식점 리뷰:",
    period_business_review_df.shape
)

print(
    period_business_review_df[
        "period"
    ].value_counts()
)

코호트 기간 음식점 리뷰: (897078, 6)
period
baseline    523100
recent      373978
Name: count, dtype: int64


In [66]:
# 16-6. 사용자·기간별 고유 음식점 집합

# 같은 기간에 같은 음식점을 여러 번 리뷰해도 음식점 수는 1개로 센다.

sample_business_period_df = (
    period_business_review_df
    .sort_values("date")
    .drop_duplicates(
        subset=[
            "sample_id",
            "period",
            "business_id"
        ],
        keep="first"
    )
    [
        [
            "sample_id",
            "user_id",
            "selection_year",
            "period",
            "business_id",
            "date"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

business_set_df = (
    sample_business_period_df
    .groupby(
        [
            "sample_id",
            "period"
        ],
        as_index=False
    )
    .agg(
        business_set=(
            "business_id",
            lambda values:
                set(values)
        )
    )
)


In [67]:
baseline_business_set_df = (
    business_set_df[
        business_set_df[
            "period"
        ]
        == "baseline"
    ]
    [
        [
            "sample_id",
            "business_set"
        ]
    ]
    .rename(
        columns={
            "business_set":
                "baseline_business_set"
        }
    )
)

recent_business_set_df = (
    business_set_df[
        business_set_df[
            "period"
        ]
        == "recent"
    ]
    [
        [
            "sample_id",
            "business_set"
        ]
    ]
    .rename(
        columns={
            "business_set":
                "recent_business_set"
        }
    )
)

In [68]:
# 16-7. 고유 음식점·재방문 피처 생성
business_feature_work_df = (
    master_cohort_df[
        [
            "sample_id",
            "user_id",
            "selection_year"
        ]
    ]
    .merge(
        baseline_business_set_df,
        on="sample_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        recent_business_set_df,
        on="sample_id",
        how="left",
        validate="one_to_one"
    )
)

In [69]:
assert business_feature_work_df[
    [
        "baseline_business_set",
        "recent_business_set"
    ]
].isna().sum().sum() == 0

In [70]:
business_feature_work_df[
    "baseline_unique_business_count"
] = (
    business_feature_work_df[
        "baseline_business_set"
    ]
    .apply(len)
)

business_feature_work_df[
    "recent_unique_business_count"
] = (
    business_feature_work_df[
        "recent_business_set"
    ]
    .apply(len)
)

In [71]:
business_feature_work_df[
    "recent_revisited_business_count"
] = (
    business_feature_work_df
    .apply(
        lambda row:
            len(
                row[
                    "baseline_business_set"
                ]
                & row[
                    "recent_business_set"
                ]
            ),
        axis=1
    )
)

In [72]:
business_feature_work_df[
    "recent_new_vs_baseline_count"
] = (
    business_feature_work_df
    .apply(
        lambda row:
            len(
                row[
                    "recent_business_set"
                ]
                - row[
                    "baseline_business_set"
                ]
            ),
        axis=1
    )
)

In [73]:
business_feature_work_df[
    "unique_business_count_diff"
] = (
    business_feature_work_df[
        "recent_unique_business_count"
    ]
    - business_feature_work_df[
        "baseline_unique_business_count"
    ]
)

business_feature_work_df[
    "unique_business_ratio"
] = (
    business_feature_work_df[
        "recent_unique_business_count"
    ]
    / business_feature_work_df[
        "baseline_unique_business_count"
    ]
)

business_feature_work_df[
    "unique_business_decline_rate"
] = (
    (
        business_feature_work_df[
            "baseline_unique_business_count"
        ]
        - business_feature_work_df[
            "recent_unique_business_count"
        ]
    )
    / business_feature_work_df[
        "baseline_unique_business_count"
    ]
)

business_feature_work_df[
    "recent_revisit_rate"
] = (
    business_feature_work_df[
        "recent_revisited_business_count"
    ]
    / business_feature_work_df[
        "recent_unique_business_count"
    ]
)

business_feature_work_df[
    "recent_new_vs_baseline_rate"
] = (
    business_feature_work_df[
        "recent_new_vs_baseline_count"
    ]
    / business_feature_work_df[
        "recent_unique_business_count"
    ]
)

In [74]:
# 16-8. 전체 과거 이력 기준 신규 음식점 피처

# 여기서 신규 음식점은 사용자가 Yelp 데이터에서 처음 리뷰한 음식점을 의미한다.
sample_business_history_df = (
    sample_business_period_df
    .merge(
        user_business_first_review_df[
            [
                "user_id",
                "business_id",
                "first_review_year"
            ]
        ],
        on=[
            "user_id",
            "business_id"
        ],
        how="left",
        validate="many_to_one"
    )
)

sample_business_history_df[
    "review_year"
] = (
    sample_business_history_df[
        "date"
    ]
    .dt.year
    .astype("int16")
)

sample_business_history_df[
    "is_new_business"
] = (
    sample_business_history_df[
        "review_year"
    ]
    == sample_business_history_df[
        "first_review_year"
    ]
).astype("int8")

In [75]:
new_business_period_df = (
    sample_business_history_df
    .groupby(
        [
            "sample_id",
            "period"
        ],
        as_index=False
    )
    .agg(
        new_business_count=(
            "is_new_business",
            "sum"
        )
    )
)

In [76]:
baseline_new_business_df = (
    new_business_period_df[
        new_business_period_df[
            "period"
        ]
        == "baseline"
    ]
    [
        [
            "sample_id",
            "new_business_count"
        ]
    ]
    .rename(
        columns={
            "new_business_count":
                "baseline_new_business_count"
        }
    )
)

recent_new_business_df = (
    new_business_period_df[
        new_business_period_df[
            "period"
        ]
        == "recent"
    ]
    [
        [
            "sample_id",
            "new_business_count"
        ]
    ]
    .rename(
        columns={
            "new_business_count":
                "recent_new_business_count"
        }
    )
)

In [77]:
business_feature_work_df = (
    business_feature_work_df
    .merge(
        baseline_new_business_df,
        on="sample_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        recent_new_business_df,
        on="sample_id",
        how="left",
        validate="one_to_one"
    )
)

business_feature_work_df[
    [
        "baseline_new_business_count",
        "recent_new_business_count"
    ]
] = (
    business_feature_work_df[
        [
            "baseline_new_business_count",
            "recent_new_business_count"
        ]
    ]
    .fillna(0)
    .astype("int32")
)

In [78]:
business_feature_work_df[
    "baseline_new_business_rate"
] = (
    business_feature_work_df[
        "baseline_new_business_count"
    ]
    / business_feature_work_df[
        "baseline_unique_business_count"
    ]
)

business_feature_work_df[
    "recent_new_business_rate"
] = (
    business_feature_work_df[
        "recent_new_business_count"
    ]
    / business_feature_work_df[
        "recent_unique_business_count"
    ]
)

business_feature_work_df[
    "new_business_count_diff"
] = (
    business_feature_work_df[
        "recent_new_business_count"
    ]
    - business_feature_work_df[
        "baseline_new_business_count"
    ]
)

business_feature_work_df[
    "new_business_rate_decline"
] = (
    business_feature_work_df[
        "baseline_new_business_rate"
    ]
    - business_feature_work_df[
        "recent_new_business_rate"
    ]
)

In [79]:
# 16-9. 최종 피처 테이블

# 집합 컬럼은 Parquet 피처 파일에 저장하지 않는다.
business_feature_df = (
    business_feature_work_df
    .drop(
        columns=[
            "baseline_business_set",
            "recent_business_set"
        ]
    )
)

In [80]:
print(
    "음식점 피처 크기:",
    business_feature_df.shape
)

business_feature_df.head()

음식점 피처 크기: (21601, 18)


,sample_id,user_id,selection_year,baseline_unique_business_count,recent_unique_business_count,recent_revisited_business_count,recent_new_vs_baseline_count,unique_business_count_diff,unique_business_ratio,unique_business_decline_rate,recent_revisit_rate,recent_new_vs_baseline_rate,baseline_new_business_count,recent_new_business_count,baseline_new_business_rate,recent_new_business_rate,new_business_count_diff,new_business_rate_decline
0,-KICU2HksIrtaOymb_jqPQ_2009,-KICU2HksIrtaOymb_jqPQ,2009,23,2,0,2,-21,0.086957,0.913043,0.000000,1.000000,23,2,1.0,1.000000,-21,0.000000
1,-VuSUcCZCbQcdSdF7w9USg_2009,-VuSUcCZCbQcdSdF7w9USg,2009,11,2,0,2,-9,0.181818,0.818182,0.000000,1.000000,11,2,1.0,1.000000,-9,0.000000
2,-hKniZN2OdshWLHYuj21jQ_2009,-hKniZN2OdshWLHYuj21jQ,2009,12,19,0,19,7,1.583333,-0.583333,0.000000,1.000000,12,19,1.0,1.000000,7,0.000000
3,-ju8d9NY3yZyVJCdw5oIUw_2009,-ju8d9NY3yZyVJCdw5oIUw,2009,10,6,0,6,-4,0.600000,0.400000,0.000000,1.000000,9,6,0.9,1.000000,-3,-0.100000
4,-y-R9jOTso_XAjDOrabdFg_2009,-y-R9jOTso_XAjDOrabdFg,2009,20,15,1,14,-5,0.750000,0.250000,0.066667,0.933333,20,14,1.0,0.933333,-6,0.066667


In [81]:
# 16-10. 논리 검증
business_feature_columns = [
    column
    for column
    in business_feature_df.columns
    if column
    not in {
        "sample_id",
        "user_id",
        "selection_year"
    }
]

business_feature_validation_df = pd.DataFrame(
    {
        "feature":
            business_feature_columns,
        "missing_count": [
            business_feature_df[
                column
            ].isna().sum()
            for column
            in business_feature_columns
        ],
        "infinite_count": [
            np.isinf(
                business_feature_df[
                    column
                ].astype(float)
            ).sum()
            for column
            in business_feature_columns
        ]
    }
)

business_feature_validation_df


,feature,missing_count,infinite_count
0,baseline_unique_business_count,0,0
1,recent_unique_business_count,0,0
2,recent_revisited_business_count,0,0
3,recent_new_vs_baseline_count,0,0
4,unique_business_count_diff,0,0
5,unique_business_ratio,0,0
6,unique_business_decline_rate,0,0
7,recent_revisit_rate,0,0
8,recent_new_vs_baseline_rate,0,0
9,baseline_new_business_count,0,0


In [82]:
assert len(
    business_feature_df
) == len(
    master_cohort_df
)

assert business_feature_df[
    "sample_id"
].is_unique

assert (
    business_feature_validation_df[
        "missing_count"
    ].sum()
    == 0
)

assert (
    business_feature_validation_df[
        "infinite_count"
    ].sum()
    == 0
)

assert (
    business_feature_df[
        "baseline_unique_business_count"
    ]
    >= 1
).all()

assert (
    business_feature_df[
        "recent_unique_business_count"
    ]
    >= 1
).all()

assert (
    business_feature_df[
        "recent_revisited_business_count"
    ]
    + business_feature_df[
        "recent_new_vs_baseline_count"
    ]
    == business_feature_df[
        "recent_unique_business_count"
    ]
).all()

assert (
    business_feature_df[
        "baseline_new_business_count"
    ]
    <= business_feature_df[
        "baseline_unique_business_count"
    ]
).all()

assert (
    business_feature_df[
        "recent_new_business_count"
    ]
    <= business_feature_df[
        "recent_unique_business_count"
    ]
).all()

print("롤링 음식점 피처 논리 검증 통과")

롤링 음식점 피처 논리 검증 통과


In [83]:
business_activity_validation_df = (
    business_feature_df[
        [
            "sample_id",
            "baseline_unique_business_count",
            "recent_unique_business_count"
        ]
    ]
    .merge(
        activity_feature_df[
            [
                "sample_id",
                "baseline_review_count",
                "recent_review_count"
            ]
        ],
        on="sample_id",
        how="left",
        validate="one_to_one"
    )
)

assert (
    business_activity_validation_df[
        "baseline_unique_business_count"
    ]
    <= business_activity_validation_df[
        "baseline_review_count"
    ]
).all()

assert (
    business_activity_validation_df[
        "recent_unique_business_count"
    ]
    <= business_activity_validation_df[
        "recent_review_count"
    ]
).all()

print("리뷰 수·고유 음식점 수 관계 검증 통과")

리뷰 수·고유 음식점 수 관계 검증 통과


In [84]:
# 16-11. 이탈·유지 비교
business_analysis_df = (
    business_feature_df
    .merge(
        master_cohort_df[
            [
                "sample_id",
                "churn"
            ]
        ],
        on="sample_id",
        how="left",
        validate="one_to_one"
    )
)

business_churn_summary_df = (
    business_analysis_df
    .groupby(
        "churn",
        as_index=False
    )
    .agg(
        samples=(
            "sample_id",
            "size"
        ),
        baseline_unique_business_mean=(
            "baseline_unique_business_count",
            "mean"
        ),
        recent_unique_business_mean=(
            "recent_unique_business_count",
            "mean"
        ),
        unique_business_decline_rate_mean=(
            "unique_business_decline_rate",
            "mean"
        ),
        recent_revisit_rate_mean=(
            "recent_revisit_rate",
            "mean"
        ),
        baseline_new_business_rate_mean=(
            "baseline_new_business_rate",
            "mean"
        ),
        recent_new_business_rate_mean=(
            "recent_new_business_rate",
            "mean"
        ),
        new_business_rate_decline_mean=(
            "new_business_rate_decline",
            "mean"
        )
    )
)

business_churn_summary_df

,churn,samples,baseline_unique_business_mean,recent_unique_business_mean,unique_business_decline_rate_mean,recent_revisit_rate_mean,baseline_new_business_rate_mean,recent_new_business_rate_mean,new_business_rate_decline_mean
0,0,18215,24.194894,18.463135,0.163424,0.039791,0.966073,0.939263,0.026810
1,1,3386,19.104843,7.951861,0.549797,0.043217,0.978881,0.944169,0.034712


In [85]:
# 16-12. 저장
business_feature_df.to_parquet(
    BUSINESS_FEATURE_PATH,
    index=False
)

business_feature_validation_df.to_csv(
    REPORT_TABLE_DIR
    / "rolling_business_feature_validation_v02.csv",
    index=False,
    encoding="utf-8-sig"
)

business_churn_summary_df.to_csv(
    REPORT_TABLE_DIR
    / "rolling_business_churn_summary_v02.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "음식점 피처 저장:",
    BUSINESS_FEATURE_PATH
)

음식점 피처 저장: C:\Users\playdata2\SKN34-2nd-5Team\data\interim\features_rolling\business_features_rolling_v02.parquet


In [87]:
# 17-0. 롤링 피처 저장 폴더 설정

from pathlib import Path

# 현재 위치부터 상위 폴더를 탐색하여 프로젝트 루트 검색
current_path = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        path
        for path in [
            current_path,
            *current_path.parents
        ]
        if (
            path
            / "data"
            / "interim"
        ).exists()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "data/interim 폴더를 기준으로 "
        "프로젝트 루트를 찾지 못했습니다."
    )

# 주요 폴더 경로
INTERIM_DIR = (
    PROJECT_ROOT
    / "data"
    / "interim"
)

FEATURE_ROLLING_DIR = (
    INTERIM_DIR
    / "features_rolling"
)

REPORT_TABLE_DIR = (
    PROJECT_ROOT
    / "reports"
    / "tables"
)

# 저장 폴더가 없으면 생성
FEATURE_ROLLING_DIR.mkdir(
    parents=True,
    exist_ok=True
)

REPORT_TABLE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(
    "프로젝트 루트:",
    PROJECT_ROOT
)

print(
    "롤링 피처 폴더:",
    FEATURE_ROLLING_DIR
)

print(
    "보고서 테이블 폴더:",
    REPORT_TABLE_DIR
)

프로젝트 루트: C:\Users\playdata2\SKN34-2nd-5Team
롤링 피처 폴더: C:\Users\playdata2\SKN34-2nd-5Team\data\interim\features_rolling
보고서 테이블 폴더: C:\Users\playdata2\SKN34-2nd-5Team\reports\tables


In [88]:
# 17-1. 카테고리 피처 경로

CATEGORY_FEATURE_PATH = (
    FEATURE_ROLLING_DIR
    / "category_features_rolling_v02.parquet"
)

CATEGORY_VALIDATION_PATH = (
    REPORT_TABLE_DIR
    / "rolling_category_feature_validation_v02.csv"
)

CATEGORY_CHURN_SUMMARY_PATH = (
    REPORT_TABLE_DIR
    / "rolling_category_churn_summary_v02.csv"
)

RESTAURANT_BUSINESS_PATH = (
    INTERIM_DIR
    / "restaurant_businesses.parquet"
)

ADDITIONAL_CULINARY_BUSINESS_PATH = (
    INTERIM_DIR
    / "additional_culinary_businesses_v02.parquet"
)

print(
    "Restaurants 업체:",
    RESTAURANT_BUSINESS_PATH.exists()
)

print(
    "미식 방문형 추가 업체:",
    ADDITIONAL_CULINARY_BUSINESS_PATH.exists()
)

Restaurants 업체: True
미식 방문형 추가 업체: True


In [89]:
# 17-2. 분석 범위에 포함되는 업체의 카테고리 불러오기

restaurant_category_df = pd.read_parquet(
    RESTAURANT_BUSINESS_PATH,
    columns=[
        "business_id",
        "categories"
    ]
)

additional_category_df = pd.read_parquet(
    ADDITIONAL_CULINARY_BUSINESS_PATH,
    columns=[
        "business_id",
        "categories"
    ]
)

business_category_df = (
    pd.concat(
        [
            restaurant_category_df,
            additional_category_df
        ],
        ignore_index=True
    )
    .drop_duplicates(
        subset=["business_id"],
        keep="first"
    )
    .reset_index(drop=True)
)

print(
    "카테고리 보유 업체:",
    len(business_category_df)
)

print(
    "고유 업체:",
    business_category_df["business_id"].nunique()
)

print(
    "카테고리 결측 업체:",
    business_category_df["categories"].isna().sum()
)

assert business_category_df[
    "business_id"
].is_unique

카테고리 보유 업체: 58156
고유 업체: 58156
카테고리 결측 업체: 0


In [90]:
# 17-3. categories 컬럼을 카테고리 목록으로 변환

def parse_category_list(value):
    if isinstance(
        value,
        (list, tuple, set, np.ndarray)
    ):
        return [
            str(category).strip()
            for category in value
            if str(category).strip()
        ]

    if value is None:
        return []

    if isinstance(value, float) and np.isnan(value):
        return []

    return [
        category.strip()
        for category in str(value).split(",")
        if category.strip()
    ]


business_category_df[
    "category_list"
] = business_category_df[
    "categories"
].apply(parse_category_list)

business_category_long_df = (
    business_category_df[
        [
            "business_id",
            "category_list"
        ]
    ]
    .explode("category_list")
    .rename(
        columns={
            "category_list": "category"
        }
    )
    .dropna(
        subset=["category"]
    )
    .drop_duplicates(
        subset=[
            "business_id",
            "category"
        ]
    )
    .reset_index(drop=True)
)

print(
    "업체-카테고리 연결 행:",
    len(business_category_long_df)
)

print(
    "고유 세부 카테고리:",
    business_category_long_df[
        "category"
    ].nunique()
)

business_category_long_df.head()


업체-카테고리 연결 행: 264574
고유 세부 카테고리: 770


,business_id,category
0,MTSW4McQd7CbVtyjqoe9mw,Restaurants
1,MTSW4McQd7CbVtyjqoe9mw,Food
2,MTSW4McQd7CbVtyjqoe9mw,Bubble Tea
3,MTSW4McQd7CbVtyjqoe9mw,Coffee & Tea
4,MTSW4McQd7CbVtyjqoe9mw,Bakeries


In [91]:
# 17-4. 표본·기간·업체와 카테고리 연결

required_columns = {
    "sample_id",
    "period",
    "business_id"
}

assert required_columns.issubset(
    sample_business_period_df.columns
)

sample_category_df = (
    sample_business_period_df[
        [
            "sample_id",
            "period",
            "business_id"
        ]
    ]
    .merge(
        business_category_long_df,
        on="business_id",
        how="left",
        validate="many_to_many"
    )
)

missing_category_rows = (
    sample_category_df[
        "category"
    ].isna().sum()
)

print(
    "카테고리 연결 실패 행:",
    missing_category_rows
)

assert missing_category_rows == 0

# 같은 음식점의 같은 카테고리는 한 기간에 한 번만 집계
sample_category_df = (
    sample_category_df
    .drop_duplicates(
        subset=[
            "sample_id",
            "period",
            "business_id",
            "category"
        ]
    )
    .reset_index(drop=True)
)

print(
    "표본-기간-업체-카테고리 행:",
    len(sample_category_df)
)

카테고리 연결 실패 행: 0
표본-기간-업체-카테고리 행: 4678239


In [92]:
# 17-5. 표본·기간·카테고리별 음식점 수

sample_category_count_df = (
    sample_category_df
    .groupby(
        [
            "sample_id",
            "period",
            "category"
        ],
        as_index=False
    )
    .agg(
        category_business_count=(
            "business_id",
            "nunique"
        )
    )
)

sample_category_count_df[
    "total_category_business_count"
] = (
    sample_category_count_df
    .groupby(
        [
            "sample_id",
            "period"
        ]
    )[
        "category_business_count"
    ]
    .transform("sum")
)

sample_category_count_df[
    "category_share"
] = (
    sample_category_count_df[
        "category_business_count"
    ]
    / sample_category_count_df[
        "total_category_business_count"
    ]
)

sample_category_count_df[
    "entropy_component"
] = -(
    sample_category_count_df[
        "category_share"
    ]
    * np.log(
        sample_category_count_df[
            "category_share"
        ]
    )
)

sample_category_count_df[
    "squared_share"
] = (
    sample_category_count_df[
        "category_share"
    ] ** 2
)

sample_category_count_df.head()

,sample_id,period,category,category_business_count,total_category_business_count,category_share,entropy_component,squared_share
0,--Vu3Gux9nPnLcG9yO_HxA_2017,baseline,American (New),4,125,0.032,0.110145,0.001024
1,--Vu3Gux9nPnLcG9yO_HxA_2017,baseline,American (Traditional),3,125,0.024,0.089513,0.000576
2,--Vu3Gux9nPnLcG9yO_HxA_2017,baseline,Bagels,1,125,0.008,0.038627,0.000064
3,--Vu3Gux9nPnLcG9yO_HxA_2017,baseline,Bakeries,2,125,0.016,0.066163,0.000256
4,--Vu3Gux9nPnLcG9yO_HxA_2017,baseline,Barbeque,1,125,0.008,0.038627,0.000064


In [93]:
# 17-6. 고유 카테고리 수·엔트로피·Simpson 다양성 계산

category_period_feature_df = (
    sample_category_count_df
    .groupby(
        [
            "sample_id",
            "period"
        ],
        as_index=False
    )
    .agg(
        unique_category_count=(
            "category",
            "nunique"
        ),
        category_entropy=(
            "entropy_component",
            "sum"
        ),
        squared_share_sum=(
            "squared_share",
            "sum"
        ),
        top_category_share=(
            "category_share",
            "max"
        )
    )
)

category_period_feature_df[
    "normalized_category_entropy"
] = np.where(
    category_period_feature_df[
        "unique_category_count"
    ] > 1,
    (
        category_period_feature_df[
            "category_entropy"
        ]
        / np.log(
            category_period_feature_df[
                "unique_category_count"
            ]
        )
    ),
    0.0
)

category_period_feature_df[
    "simpson_category_diversity"
] = (
    1
    - category_period_feature_df[
        "squared_share_sum"
    ]
)

category_period_feature_df = (
    category_period_feature_df[
        [
            "sample_id",
            "period",
            "unique_category_count",
            "normalized_category_entropy",
            "simpson_category_diversity",
            "top_category_share"
        ]
    ]
)

category_period_feature_df.head()

,sample_id,period,unique_category_count,normalized_category_entropy,simpson_category_diversity,top_category_share
0,--Vu3Gux9nPnLcG9yO_HxA_2017,baseline,47,0.869911,0.941696,0.144000
1,--Vu3Gux9nPnLcG9yO_HxA_2017,recent,15,0.952576,0.908587,0.210526
2,--u09WAjW741FdfkJXxNmg_2017,baseline,46,0.890242,0.950414,0.138462
3,--u09WAjW741FdfkJXxNmg_2017,recent,35,0.921366,0.945502,0.161765
4,-00kdEIhCt-ODaV4BS-EAg_2012,baseline,27,0.875245,0.915359,0.210526


In [95]:
# 롤링 마스터 코호트 다시 불러오기

ROLLING_COHORT_PATH = (
    INTERIM_DIR
    / "rolling"
    / "culinary_rolling_cohort_master_v02.parquet"
)

if not ROLLING_COHORT_PATH.exists():
    raise FileNotFoundError(
        f"롤링 코호트 파일을 찾을 수 없습니다:\n"
        f"{ROLLING_COHORT_PATH}"
    )

rolling_cohort_df = pd.read_parquet(
    ROLLING_COHORT_PATH
)

required_cohort_columns = {
    "sample_id",
    "user_id",
    "selection_year",
    "churn"
}

missing_cohort_columns = (
    required_cohort_columns
    - set(rolling_cohort_df.columns)
)

if missing_cohort_columns:
    raise KeyError(
        "마스터 코호트에 필요한 컬럼이 없습니다: "
        f"{sorted(missing_cohort_columns)}"
    )

print(
    "롤링 코호트 경로:",
    ROLLING_COHORT_PATH
)

print(
    "롤링 코호트 크기:",
    rolling_cohort_df.shape
)

print(
    "고유 표본:",
    rolling_cohort_df[
        "sample_id"
    ].nunique()
)

print(
    "표본 중복:",
    rolling_cohort_df[
        "sample_id"
    ].duplicated().sum()
)

display(
    rolling_cohort_df[
        [
            "sample_id",
            "user_id",
            "selection_year",
            "churn"
        ]
    ].head()
)

assert len(rolling_cohort_df) == 21_601
assert rolling_cohort_df["sample_id"].is_unique
assert set(
    rolling_cohort_df["churn"].unique()
).issubset({0, 1})

롤링 코호트 경로: C:\Users\playdata2\SKN34-2nd-5Team\data\interim\rolling\culinary_rolling_cohort_master_v02.parquet
롤링 코호트 크기: (21601, 13)
고유 표본: 21601
표본 중복: 0


,sample_id,user_id,selection_year,churn
0,-KICU2HksIrtaOymb_jqPQ_2009,-KICU2HksIrtaOymb_jqPQ,2009,0
1,-VuSUcCZCbQcdSdF7w9USg_2009,-VuSUcCZCbQcdSdF7w9USg,2009,1
2,-hKniZN2OdshWLHYuj21jQ_2009,-hKniZN2OdshWLHYuj21jQ,2009,0
3,-ju8d9NY3yZyVJCdw5oIUw_2009,-ju8d9NY3yZyVJCdw5oIUw,2009,0
4,-y-R9jOTso_XAjDOrabdFg_2009,-y-R9jOTso_XAjDOrabdFg,2009,0


In [96]:
# 17-7. Baseline 카테고리 피처

baseline_category_df = (
    category_period_feature_df[
        category_period_feature_df[
            "period"
        ].eq("baseline")
    ]
    .drop(
        columns=["period"]
    )
    .rename(
        columns={
            "unique_category_count":
                "baseline_unique_category_count",
            "normalized_category_entropy":
                "baseline_normalized_category_entropy",
            "simpson_category_diversity":
                "baseline_simpson_category_diversity",
            "top_category_share":
                "baseline_top_category_share"
        }
    )
)

# Recent 카테고리 피처
recent_category_df = (
    category_period_feature_df[
        category_period_feature_df[
            "period"
        ].eq("recent")
    ]
    .drop(
        columns=["period"]
    )
    .rename(
        columns={
            "unique_category_count":
                "recent_unique_category_count",
            "normalized_category_entropy":
                "recent_normalized_category_entropy",
            "simpson_category_diversity":
                "recent_simpson_category_diversity",
            "top_category_share":
                "recent_top_category_share"
        }
    )
)

category_feature_df = (
    rolling_cohort_df[
        [
            "sample_id",
            "user_id",
            "selection_year",
            "churn"
        ]
    ]
    .merge(
        baseline_category_df,
        on="sample_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        recent_category_df,
        on="sample_id",
        how="left",
        validate="one_to_one"
    )
)

print(
    "카테고리 피처 기본 크기:",
    category_feature_df.shape
)

카테고리 피처 기본 크기: (21601, 12)


In [97]:
# 17-8. 카테고리 변화 피처

category_feature_df[
    "unique_category_count_diff"
] = (
    category_feature_df[
        "recent_unique_category_count"
    ]
    - category_feature_df[
        "baseline_unique_category_count"
    ]
)

category_feature_df[
    "unique_category_ratio"
] = (
    category_feature_df[
        "recent_unique_category_count"
    ]
    / category_feature_df[
        "baseline_unique_category_count"
    ]
)

category_feature_df[
    "unique_category_decline_rate"
] = (
    1
    - category_feature_df[
        "unique_category_ratio"
    ]
)

category_feature_df[
    "category_entropy_decline"
] = (
    category_feature_df[
        "baseline_normalized_category_entropy"
    ]
    - category_feature_df[
        "recent_normalized_category_entropy"
    ]
)

category_feature_df[
    "simpson_diversity_decline"
] = (
    category_feature_df[
        "baseline_simpson_category_diversity"
    ]
    - category_feature_df[
        "recent_simpson_category_diversity"
    ]
)

category_feature_df[
    "top_category_share_increase"
] = (
    category_feature_df[
        "recent_top_category_share"
    ]
    - category_feature_df[
        "baseline_top_category_share"
    ]
)

In [98]:
# 17-9. 카테고리 피처 논리 검증

category_feature_columns = [
    column
    for column in category_feature_df.columns
    if column not in [
        "sample_id",
        "user_id",
        "selection_year",
        "churn"
    ]
]

category_validation_df = pd.DataFrame({
    "feature": category_feature_columns,
    "missing_count": [
        category_feature_df[
            column
        ].isna().sum()
        for column in category_feature_columns
    ],
    "infinite_count": [
        np.isinf(
            category_feature_df[
                column
            ]
        ).sum()
        for column in category_feature_columns
    ]
})

display(category_validation_df)

assert category_feature_df[
    "sample_id"
].is_unique

assert len(category_feature_df) == len(
    rolling_cohort_df
)

assert category_validation_df[
    "missing_count"
].sum() == 0

assert category_validation_df[
    "infinite_count"
].sum() == 0

# 카테고리 수는 최소 1개 이상
assert (
    category_feature_df[
        "baseline_unique_category_count"
    ] >= 1
).all()

assert (
    category_feature_df[
        "recent_unique_category_count"
    ] >= 1
).all()

# 비율형 다양성 지표 범위 검증
bounded_columns = [
    "baseline_normalized_category_entropy",
    "recent_normalized_category_entropy",
    "baseline_simpson_category_diversity",
    "recent_simpson_category_diversity",
    "baseline_top_category_share",
    "recent_top_category_share"
]

tolerance = 1e-9

for column in bounded_columns:
    assert (
        category_feature_df[column]
        >= -tolerance
    ).all()

    assert (
        category_feature_df[column]
        <= 1 + tolerance
    ).all()

print("롤링 카테고리 피처 검증 통과")

,feature,missing_count,infinite_count
0,baseline_unique_category_count,0,0
1,baseline_normalized_category_entropy,0,0
2,baseline_simpson_category_diversity,0,0
3,baseline_top_category_share,0,0
4,recent_unique_category_count,0,0
5,recent_normalized_category_entropy,0,0
6,recent_simpson_category_diversity,0,0
7,recent_top_category_share,0,0
8,unique_category_count_diff,0,0
9,unique_category_ratio,0,0


롤링 카테고리 피처 검증 통과


In [99]:
# 17-10. 이탈 여부별 카테고리 피처 비교

category_churn_summary_df = (
    category_feature_df
    .groupby(
        "churn",
        as_index=False
    )
    .agg(
        samples=(
            "sample_id",
            "size"
        ),
        baseline_unique_category_mean=(
            "baseline_unique_category_count",
            "mean"
        ),
        recent_unique_category_mean=(
            "recent_unique_category_count",
            "mean"
        ),
        unique_category_decline_rate_mean=(
            "unique_category_decline_rate",
            "mean"
        ),
        baseline_entropy_mean=(
            "baseline_normalized_category_entropy",
            "mean"
        ),
        recent_entropy_mean=(
            "recent_normalized_category_entropy",
            "mean"
        ),
        category_entropy_decline_mean=(
            "category_entropy_decline",
            "mean"
        ),
        simpson_diversity_decline_mean=(
            "simpson_diversity_decline",
            "mean"
        ),
        top_category_share_increase_mean=(
            "top_category_share_increase",
            "mean"
        )
    )
)

display(category_churn_summary_df)

,churn,samples,baseline_unique_category_mean,recent_unique_category_mean,unique_category_decline_rate_mean,baseline_entropy_mean,recent_entropy_mean,category_entropy_decline_mean,simpson_diversity_decline_mean,top_category_share_increase_mean
0,0,18215,46.58924,38.288938,0.144923,0.881204,0.903315,-0.022111,0.011877,0.001447
1,1,3386,41.67218,22.645895,0.431448,0.889541,0.939757,-0.050216,0.040440,0.010855


In [100]:
# 17-11. 활동량 감소와 카테고리 감소의 상관관계

category_activity_correlation_df = (
    activity_feature_df[
        [
            "sample_id",
            "review_count_decline_rate"
        ]
    ]
    .merge(
        category_feature_df[
            [
                "sample_id",
                "unique_category_decline_rate",
                "category_entropy_decline",
                "simpson_diversity_decline",
                "top_category_share_increase"
            ]
        ],
        on="sample_id",
        how="inner",
        validate="one_to_one"
    )
    .drop(
        columns=["sample_id"]
    )
    .corr()
    .round(3)
)

display(category_activity_correlation_df)

,review_count_decline_rate,unique_category_decline_rate,category_entropy_decline,simpson_diversity_decline,top_category_share_increase
review_count_decline_rate,1.000,0.837,-0.743,0.315,0.116
unique_category_decline_rate,0.837,1.000,-0.687,0.549,0.379
category_entropy_decline,-0.743,-0.687,1.000,-0.364,0.022
simpson_diversity_decline,0.315,0.549,-0.364,1.000,0.777
top_category_share_increase,0.116,0.379,0.022,0.777,1.000


In [101]:
# 17-12. 카테고리 피처와 검증 결과 저장

FEATURE_ROLLING_DIR.mkdir(
    parents=True,
    exist_ok=True
)

REPORT_TABLE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

category_feature_df.to_parquet(
    CATEGORY_FEATURE_PATH,
    index=False
)

category_validation_df.to_csv(
    CATEGORY_VALIDATION_PATH,
    index=False,
    encoding="utf-8-sig"
)

category_churn_summary_df.to_csv(
    CATEGORY_CHURN_SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig"
)

print(
    "카테고리 피처 저장:",
    CATEGORY_FEATURE_PATH
)

print(
    "카테고리 피처 크기:",
    category_feature_df.shape
)

카테고리 피처 저장: C:\Users\playdata2\SKN34-2nd-5Team\data\interim\features_rolling\category_features_rolling_v02.parquet
카테고리 피처 크기: (21601, 18)


In [102]:
# 18. 롤링 탐방 반경 피처
# 18-1. 탐방 반경 피처 저장 경로

SPATIAL_FEATURE_PATH = (
    FEATURE_ROLLING_DIR
    / "spatial_features_rolling_v02.parquet"
)

SPATIAL_VALIDATION_PATH = (
    REPORT_TABLE_DIR
    / "rolling_spatial_feature_validation_v02.csv"
)

SPATIAL_CHURN_SUMMARY_PATH = (
    REPORT_TABLE_DIR
    / "rolling_spatial_churn_summary_v02.csv"
)

SPATIAL_CORRELATION_PATH = (
    REPORT_TABLE_DIR
    / "rolling_spatial_activity_correlation_v02.csv"
)

In [103]:
# 18-2. Restaurants와 미식 방문형 업체 위치 불러오기

restaurant_location_df = pd.read_parquet(
    RESTAURANT_BUSINESS_PATH,
    columns=[
        "business_id",
        "latitude",
        "longitude"
    ]
)

additional_location_df = pd.read_parquet(
    ADDITIONAL_CULINARY_BUSINESS_PATH,
    columns=[
        "business_id",
        "latitude",
        "longitude"
    ]
)

business_location_df = (
    pd.concat(
        [
            restaurant_location_df,
            additional_location_df
        ],
        ignore_index=True
    )
    .drop_duplicates(
        subset=["business_id"],
        keep="first"
    )
    .reset_index(drop=True)
)

business_location_df[
    "latitude"
] = pd.to_numeric(
    business_location_df["latitude"],
    errors="coerce"
)

business_location_df[
    "longitude"
] = pd.to_numeric(
    business_location_df["longitude"],
    errors="coerce"
)

print(
    "위치 데이터 업체:",
    len(business_location_df)
)

print(
    "고유 업체:",
    business_location_df[
        "business_id"
    ].nunique()
)

print(
    "위도 결측:",
    business_location_df[
        "latitude"
    ].isna().sum()
)

print(
    "경도 결측:",
    business_location_df[
        "longitude"
    ].isna().sum()
)

assert business_location_df[
    "business_id"
].is_unique

assert business_location_df[
    [
        "latitude",
        "longitude"
    ]
].isna().sum().sum() == 0

assert business_location_df[
    "latitude"
].between(-90, 90).all()

assert business_location_df[
    "longitude"
].between(-180, 180).all()

위치 데이터 업체: 58156
고유 업체: 58156
위도 결측: 0
경도 결측: 0


In [104]:
# 18-3. 16번에서 만든 표본별 방문 음식점에 위치 결합

required_business_period_columns = {
    "sample_id",
    "period",
    "business_id"
}

if not required_business_period_columns.issubset(
    sample_business_period_df.columns
):
    raise KeyError(
        "sample_business_period_df에 필요한 컬럼이 없습니다. "
        "16번 음식점 피처 생성 셀부터 다시 실행하세요."
    )

sample_spatial_df = (
    sample_business_period_df[
        [
            "sample_id",
            "period",
            "business_id"
        ]
    ]
    .merge(
        business_location_df,
        on="business_id",
        how="left",
        validate="many_to_one"
    )
)

spatial_merge_validation_df = pd.DataFrame({
    "metric": [
        "total_rows",
        "missing_latitude_rows",
        "missing_longitude_rows"
    ],
    "value": [
        len(sample_spatial_df),
        sample_spatial_df[
            "latitude"
        ].isna().sum(),
        sample_spatial_df[
            "longitude"
        ].isna().sum()
    ]
})

display(spatial_merge_validation_df)

assert sample_spatial_df[
    [
        "latitude",
        "longitude"
    ]
].isna().sum().sum() == 0

,metric,value
0,total_rows,868630
1,missing_latitude_rows,0
2,missing_longitude_rows,0


In [105]:
# 18-4. 표본·기간별 중앙 탐방 지점 계산

sample_spatial_df[
    "center_latitude"
] = (
    sample_spatial_df
    .groupby(
        [
            "sample_id",
            "period"
        ]
    )[
        "latitude"
    ]
    .transform("median")
)

sample_spatial_df[
    "center_longitude"
] = (
    sample_spatial_df
    .groupby(
        [
            "sample_id",
            "period"
        ]
    )[
        "longitude"
    ]
    .transform("median")
)

sample_spatial_df.head()

,sample_id,period,business_id,latitude,longitude,center_latitude,center_longitude
0,tgTt8j-UCJyJxvK58yDA-g_2009,baseline,8BhNur6_XKLjCDvbhX2CNg,29.958158,-90.065161,29.962765,-90.068256
1,pbDJGS8PDgqpUPmDH1YJvA_2009,baseline,ubeDltaADmtgyxkrObkOfw,39.948550,-75.144734,39.949335,-75.152761
2,tgTt8j-UCJyJxvK58yDA-g_2009,baseline,0g9fGg7Z368fPJiBp7JvdA,29.954798,-90.069150,29.962765,-90.068256
3,tgTt8j-UCJyJxvK58yDA-g_2009,baseline,cIBr1KeRLTpgvTpF8z2nFQ,29.953510,-90.067547,29.962765,-90.068256
4,2vgqdk3vuAM-LTRKxI9nfA_2009,baseline,tWJ3DwufXIKdlLTEw7K96g,38.641305,-90.262250,38.646795,-90.337008


In [106]:
# 18-4. 표본·기간별 중앙 탐방 지점 계산

sample_spatial_df[
    "center_latitude"
] = (
    sample_spatial_df
    .groupby(
        [
            "sample_id",
            "period"
        ]
    )[
        "latitude"
    ]
    .transform("median")
)

sample_spatial_df[
    "center_longitude"
] = (
    sample_spatial_df
    .groupby(
        [
            "sample_id",
            "period"
        ]
    )[
        "longitude"
    ]
    .transform("median")
)

sample_spatial_df.head()

,sample_id,period,business_id,latitude,longitude,center_latitude,center_longitude
0,tgTt8j-UCJyJxvK58yDA-g_2009,baseline,8BhNur6_XKLjCDvbhX2CNg,29.958158,-90.065161,29.962765,-90.068256
1,pbDJGS8PDgqpUPmDH1YJvA_2009,baseline,ubeDltaADmtgyxkrObkOfw,39.948550,-75.144734,39.949335,-75.152761
2,tgTt8j-UCJyJxvK58yDA-g_2009,baseline,0g9fGg7Z368fPJiBp7JvdA,29.954798,-90.069150,29.962765,-90.068256
3,tgTt8j-UCJyJxvK58yDA-g_2009,baseline,cIBr1KeRLTpgvTpF8z2nFQ,29.953510,-90.067547,29.962765,-90.068256
4,2vgqdk3vuAM-LTRKxI9nfA_2009,baseline,tWJ3DwufXIKdlLTEw7K96g,38.641305,-90.262250,38.646795,-90.337008


In [107]:
# 18-5. Haversine 거리 계산 함수

def haversine_km(
    latitude_1,
    longitude_1,
    latitude_2,
    longitude_2
):
    earth_radius_km = 6371.0088

    latitude_1_rad = np.radians(latitude_1)
    longitude_1_rad = np.radians(longitude_1)
    latitude_2_rad = np.radians(latitude_2)
    longitude_2_rad = np.radians(longitude_2)

    latitude_diff = (
        latitude_2_rad
        - latitude_1_rad
    )

    longitude_diff = (
        longitude_2_rad
        - longitude_1_rad
    )

    haversine_value = (
        np.sin(
            latitude_diff / 2
        ) ** 2
        + np.cos(latitude_1_rad)
        * np.cos(latitude_2_rad)
        * np.sin(
            longitude_diff / 2
        ) ** 2
    )

    haversine_value = np.clip(
        haversine_value,
        0,
        1
    )

    return (
        2
        * earth_radius_km
        * np.arcsin(
            np.sqrt(haversine_value)
        )
    )

In [108]:
# 음식점과 기간별 탐방 중심점 사이 거리

sample_spatial_df[
    "distance_from_center_km"
] = haversine_km(
    sample_spatial_df[
        "center_latitude"
    ],
    sample_spatial_df[
        "center_longitude"
    ],
    sample_spatial_df[
        "latitude"
    ],
    sample_spatial_df[
        "longitude"
    ]
)

assert (
    sample_spatial_df[
        "distance_from_center_km"
    ] >= 0
).all()

sample_spatial_df[
    "distance_from_center_km"
].describe(
    percentiles=[
        0.50,
        0.75,
        0.90,
        0.95,
        0.99
    ]
)

count    868630.000000
mean         42.589323
std         245.595970
min           0.000000
50%           4.729653
75%          11.126771
90%          21.361658
95%          32.063996
99%        1370.692636
max        3985.058584
Name: distance_from_center_km, dtype: float64

In [109]:
# 18-6. 표본·기간별 공간 피처 집계

spatial_period_feature_df = (
    sample_spatial_df
    .groupby(
        [
            "sample_id",
            "period"
        ],
        as_index=False
    )
    .agg(
        spatial_business_count=(
            "business_id",
            "nunique"
        ),
        median_radius_km=(
            "distance_from_center_km",
            "median"
        ),
        p90_radius_km=(
            "distance_from_center_km",
            lambda series: series.quantile(0.90)
        ),
        center_latitude=(
            "center_latitude",
            "first"
        ),
        center_longitude=(
            "center_longitude",
            "first"
        )
    )
)

spatial_period_feature_df.head()

,sample_id,period,spatial_business_count,median_radius_km,p90_radius_km,center_latitude,center_longitude
0,--Vu3Gux9nPnLcG9yO_HxA_2017,baseline,22,4.006595,9.521620,39.527148,-119.808136
1,--Vu3Gux9nPnLcG9yO_HxA_2017,recent,4,0.718685,4.155118,39.526354,-119.812154
2,--u09WAjW741FdfkJXxNmg_2017,baseline,19,3.153950,9.254834,27.946160,-82.478544
3,--u09WAjW741FdfkJXxNmg_2017,recent,12,2.006415,6.096306,27.932631,-82.502050
4,-00kdEIhCt-ODaV4BS-EAg_2012,baseline,12,14.492319,34.000805,40.232502,-75.241427


In [110]:
# 18-7. Baseline 공간 피처

baseline_spatial_df = (
    spatial_period_feature_df[
        spatial_period_feature_df[
            "period"
        ].eq("baseline")
    ]
    .drop(
        columns=["period"]
    )
    .rename(
        columns={
            "spatial_business_count":
                "baseline_spatial_business_count",
            "median_radius_km":
                "baseline_median_radius_km",
            "p90_radius_km":
                "baseline_p90_radius_km",
            "center_latitude":
                "baseline_center_latitude",
            "center_longitude":
                "baseline_center_longitude"
        }
    )
)

# Recent 공간 피처
recent_spatial_df = (
    spatial_period_feature_df[
        spatial_period_feature_df[
            "period"
        ].eq("recent")
    ]
    .drop(
        columns=["period"]
    )
    .rename(
        columns={
            "spatial_business_count":
                "recent_spatial_business_count",
            "median_radius_km":
                "recent_median_radius_km",
            "p90_radius_km":
                "recent_p90_radius_km",
            "center_latitude":
                "recent_center_latitude",
            "center_longitude":
                "recent_center_longitude"
        }
    )
)

In [111]:
# 18-8. 롤링 코호트와 공간 피처 결합

spatial_feature_build_df = (
    rolling_cohort_df[
        [
            "sample_id",
            "user_id",
            "selection_year",
            "churn"
        ]
    ]
    .merge(
        baseline_spatial_df,
        on="sample_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        recent_spatial_df,
        on="sample_id",
        how="left",
        validate="one_to_one"
    )
)

# 탐방 반경 축소량
spatial_feature_build_df[
    "median_radius_decline_km"
] = (
    spatial_feature_build_df[
        "baseline_median_radius_km"
    ]
    - spatial_feature_build_df[
        "recent_median_radius_km"
    ]
)

spatial_feature_build_df[
    "p90_radius_decline_km"
] = (
    spatial_feature_build_df[
        "baseline_p90_radius_km"
    ]
    - spatial_feature_build_df[
        "recent_p90_radius_km"
    ]
)

# 장거리 이상치 영향을 줄인 로그 기반 반경 변화
spatial_feature_build_df[
    "log_p90_radius_decline"
] = (
    np.log1p(
        spatial_feature_build_df[
            "baseline_p90_radius_km"
        ]
    )
    - np.log1p(
        spatial_feature_build_df[
            "recent_p90_radius_km"
        ]
    )
)

# Baseline 중심점과 Recent 중심점 사이 이동 거리
spatial_feature_build_df[
    "center_shift_km"
] = haversine_km(
    spatial_feature_build_df[
        "baseline_center_latitude"
    ],
    spatial_feature_build_df[
        "baseline_center_longitude"
    ],
    spatial_feature_build_df[
        "recent_center_latitude"
    ],
    spatial_feature_build_df[
        "recent_center_longitude"
    ]
)

spatial_feature_build_df[
    "log_center_shift"
] = np.log1p(
    spatial_feature_build_df[
        "center_shift_km"
    ]
)

# 최근 음식점이 2곳 이상이어야 반경이 의미 있게 계산됨
spatial_feature_build_df[
    "recent_spatial_available"
] = (
    spatial_feature_build_df[
        "recent_spatial_business_count"
    ] >= 2
).astype("int8")

In [112]:
# 18-9. 최종 공간 피처 구성

spatial_feature_df = (
    spatial_feature_build_df[
        [
            "sample_id",
            "user_id",
            "selection_year",
            "churn",

            "baseline_spatial_business_count",
            "baseline_median_radius_km",
            "baseline_p90_radius_km",

            "recent_spatial_business_count",
            "recent_median_radius_km",
            "recent_p90_radius_km",

            "median_radius_decline_km",
            "p90_radius_decline_km",
            "log_p90_radius_decline",

            "center_shift_km",
            "log_center_shift",
            "recent_spatial_available"
        ]
    ]
    .copy()
)

print(
    "공간 피처 크기:",
    spatial_feature_df.shape
)

공간 피처 크기: (21601, 16)


In [113]:
# 18-10. 결측·무한대 검증

spatial_feature_columns = [
    column
    for column in spatial_feature_df.columns
    if column not in [
        "sample_id",
        "user_id",
        "selection_year",
        "churn"
    ]
]

spatial_validation_df = pd.DataFrame({
    "feature": spatial_feature_columns,
    "missing_count": [
        spatial_feature_df[
            column
        ].isna().sum()
        for column in spatial_feature_columns
    ],
    "infinite_count": [
        np.isinf(
            spatial_feature_df[
                column
            ]
        ).sum()
        for column in spatial_feature_columns
    ]
})

display(spatial_validation_df)

assert spatial_feature_df[
    "sample_id"
].is_unique

assert len(spatial_feature_df) == len(
    rolling_cohort_df
)

assert spatial_validation_df[
    "missing_count"
].sum() == 0

assert spatial_validation_df[
    "infinite_count"
].sum() == 0

non_negative_columns = [
    "baseline_median_radius_km",
    "baseline_p90_radius_km",
    "recent_median_radius_km",
    "recent_p90_radius_km",
    "center_shift_km",
    "log_center_shift"
]

for column in non_negative_columns:
    assert (
        spatial_feature_df[column] >= 0
    ).all()

assert set(
    spatial_feature_df[
        "recent_spatial_available"
    ].unique()
).issubset({0, 1})

print("롤링 탐방 반경 피처 검증 통과")

,feature,missing_count,infinite_count
0,baseline_spatial_business_count,0,0
1,baseline_median_radius_km,0,0
2,baseline_p90_radius_km,0,0
3,recent_spatial_business_count,0,0
4,recent_median_radius_km,0,0
5,recent_p90_radius_km,0,0
6,median_radius_decline_km,0,0
7,p90_radius_decline_km,0,0
8,log_p90_radius_decline,0,0
9,center_shift_km,0,0


롤링 탐방 반경 피처 검증 통과


In [114]:
# 18-11. 이탈 여부별 공간 피처 비교

spatial_churn_summary_df = (
    spatial_feature_df
    .groupby(
        "churn",
        as_index=False
    )
    .agg(
        samples=(
            "sample_id",
            "size"
        ),
        baseline_p90_radius_median=(
            "baseline_p90_radius_km",
            "median"
        ),
        recent_p90_radius_median=(
            "recent_p90_radius_km",
            "median"
        ),
        p90_radius_decline_median=(
            "p90_radius_decline_km",
            "median"
        ),
        log_p90_radius_decline_median=(
            "log_p90_radius_decline",
            "median"
        ),
        center_shift_median=(
            "center_shift_km",
            "median"
        ),
        recent_spatial_available_rate=(
            "recent_spatial_available",
            "mean"
        )
    )
)

spatial_churn_summary_df[
    "recent_spatial_available_rate"
] *= 100

display(spatial_churn_summary_df)

,churn,samples,baseline_p90_radius_median,recent_p90_radius_median,p90_radius_decline_median,log_p90_radius_decline_median,center_shift_median,recent_spatial_available_rate
0,0,18215,14.357413,13.438129,0.457101,0.035906,2.660988,97.578918
1,1,3386,12.987859,9.021300,2.512910,0.253753,3.261581,87.300650


In [115]:
# 18-12. 활동량 감소와 공간 피처 상관관계

spatial_activity_correlation_df = (
    activity_feature_df[
        [
            "sample_id",
            "review_count_decline_rate"
        ]
    ]
    .merge(
        spatial_feature_df[
            [
                "sample_id",
                "p90_radius_decline_km",
                "log_p90_radius_decline",
                "center_shift_km",
                "log_center_shift"
            ]
        ],
        on="sample_id",
        how="inner",
        validate="one_to_one"
    )
    .drop(
        columns=["sample_id"]
    )
    .corr()
    .round(3)
)

display(spatial_activity_correlation_df)

,review_count_decline_rate,p90_radius_decline_km,log_p90_radius_decline,center_shift_km,log_center_shift
review_count_decline_rate,1.000,0.056,0.165,0.048,0.104
p90_radius_decline_km,0.056,1.000,0.793,0.106,0.075
log_p90_radius_decline,0.165,0.793,1.000,0.075,0.088
center_shift_km,0.048,0.106,0.075,1.000,0.704
log_center_shift,0.104,0.075,0.088,0.704,1.000


In [116]:
# 18-13. 공간 피처와 검증 결과 저장

spatial_feature_df.to_parquet(
    SPATIAL_FEATURE_PATH,
    index=False
)

spatial_validation_df.to_csv(
    SPATIAL_VALIDATION_PATH,
    index=False,
    encoding="utf-8-sig"
)

spatial_churn_summary_df.to_csv(
    SPATIAL_CHURN_SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig"
)

spatial_activity_correlation_df.to_csv(
    SPATIAL_CORRELATION_PATH,
    encoding="utf-8-sig"
)

print(
    "공간 피처 저장:",
    SPATIAL_FEATURE_PATH
)

print(
    "공간 피처 크기:",
    spatial_feature_df.shape
)

공간 피처 저장: C:\Users\playdata2\SKN34-2nd-5Team\data\interim\features_rolling\spatial_features_rolling_v02.parquet
공간 피처 크기: (21601, 16)


In [117]:
# 19. 롤링 평점 피처
# 19-1. 평점 피처 저장 경로

RATING_FEATURE_PATH = (
    FEATURE_ROLLING_DIR
    / "rating_features_rolling_v02.parquet"
)

RATING_VALIDATION_PATH = (
    REPORT_TABLE_DIR
    / "rolling_rating_feature_validation_v02.csv"
)

RATING_CHURN_SUMMARY_PATH = (
    REPORT_TABLE_DIR
    / "rolling_rating_churn_summary_v02.csv"
)

RATING_CORRELATION_PATH = (
    REPORT_TABLE_DIR
    / "rolling_rating_activity_correlation_v02.csv"
)

RESTAURANT_REVIEW_PATH = (
    INTERIM_DIR
    / "restaurant_reviews.parquet"
)

ADDITIONAL_CULINARY_REVIEW_PATH = (
    INTERIM_DIR
    / "additional_culinary_reviews_v02.parquet"
)

print(
    "Restaurants 리뷰:",
    RESTAURANT_REVIEW_PATH.exists()
)

print(
    "미식 방문형 리뷰:",
    ADDITIONAL_CULINARY_REVIEW_PATH.exists()
)

Restaurants 리뷰: True
미식 방문형 리뷰: True


In [119]:
# 19-2. Restaurants와 미식 방문형 리뷰 결합

restaurant_rating_review_df = pd.read_parquet(
    RESTAURANT_REVIEW_PATH,
    columns=[
        "user_id",
        "stars",
        "date"
    ]
)

additional_rating_review_df = pd.read_parquet(
    ADDITIONAL_CULINARY_REVIEW_PATH,
    columns=[
        "user_id",
        "stars",
        "date"
    ]
)

rating_review_df = pd.concat(
    [
        restaurant_rating_review_df,
        additional_rating_review_df
    ],
    ignore_index=True
)

rating_review_df[
    "date"
] = pd.to_datetime(
    rating_review_df["date"],
    errors="coerce"
)

rating_review_df[
    "stars"
] = pd.to_numeric(
    rating_review_df["stars"],
    errors="coerce"
)

print(
    "전체 평점 리뷰:",
    len(rating_review_df)
)

print(
    "날짜 결측:",
    rating_review_df[
        "date"
    ].isna().sum()
)

print(
    "평점 결측:",
    rating_review_df[
        "stars"
    ].isna().sum()
)

assert rating_review_df[
    [
        "date",
        "stars"
    ]
].isna().sum().sum() == 0

assert rating_review_df[
    "stars"
].between(1, 5).all()

전체 평점 리뷰: 4950264
날짜 결측: 0
평점 결측: 0


In [120]:
# 19-3. 롤링 코호트에 포함되는 사용자만 선택

rolling_user_ids = rolling_cohort_df[
    "user_id"
].unique()

rating_review_df = (
    rating_review_df[
        rating_review_df[
            "user_id"
        ].isin(rolling_user_ids)
    ]
    .copy()
)

rating_review_df[
    "review_year"
] = rating_review_df[
    "date"
].dt.year.astype("int16")

print(
    "코호트 사용자 리뷰:",
    len(rating_review_df)
)

print(
    "코호트 리뷰 사용자:",
    rating_review_df[
        "user_id"
    ].nunique()
)

코호트 사용자 리뷰: 903036
코호트 리뷰 사용자: 12709


In [121]:
# 19-4. Baseline 기간 매핑

baseline_period_map_df = (
    rolling_cohort_df[
        [
            "sample_id",
            "user_id",
            "selection_year"
        ]
    ]
    .rename(
        columns={
            "selection_year": "review_year"
        }
    )
    .assign(
        period="baseline"
    )
)

# Recent 기간은 선정연도 다음 해
recent_period_map_df = (
    rolling_cohort_df[
        [
            "sample_id",
            "user_id",
            "selection_year"
        ]
    ]
    .assign(
        review_year=lambda frame:
            frame["selection_year"] + 1,
        period="recent"
    )
    [
        [
            "sample_id",
            "user_id",
            "review_year",
            "period"
        ]
    ]
)

baseline_period_map_df = (
    baseline_period_map_df[
        [
            "sample_id",
            "user_id",
            "review_year",
            "period"
        ]
    ]
)

sample_period_map_df = pd.concat(
    [
        baseline_period_map_df,
        recent_period_map_df
    ],
    ignore_index=True
)

print(
    "표본-기간 매핑:",
    sample_period_map_df.shape
)

assert len(sample_period_map_df) == (
    len(rolling_cohort_df) * 2
)

표본-기간 매핑: (43202, 4)


In [122]:
# 19-5. 리뷰를 해당 표본의 Baseline·Recent 기간에 연결

sample_rating_review_df = (
    rating_review_df
    .merge(
        sample_period_map_df,
        on=[
            "user_id",
            "review_year"
        ],
        how="inner",
        validate="many_to_many"
    )
)

print(
    "코호트 기간 평점 리뷰:",
    sample_rating_review_df.shape
)

display(
    sample_rating_review_df[
        "period"
    ].value_counts()
)

period_coverage_df = (
    sample_rating_review_df[
        [
            "sample_id",
            "period"
        ]
    ]
    .drop_duplicates()
)

period_coverage_count_df = (
    period_coverage_df
    .groupby(
        "sample_id"
    )[
        "period"
    ]
    .nunique()
)

assert (
    period_coverage_count_df == 2
).all()

print("모든 표본의 Baseline·Recent 리뷰 확인")

코호트 기간 평점 리뷰: (897078, 6)


period
baseline    523100
recent      373978
Name: count, dtype: int64

모든 표본의 Baseline·Recent 리뷰 확인


In [123]:
# 19-6. 저평점·고평점 여부 생성

sample_rating_review_df[
    "is_low_rating"
] = (
    sample_rating_review_df[
        "stars"
    ] <= 2
).astype("int8")

sample_rating_review_df[
    "is_high_rating"
] = (
    sample_rating_review_df[
        "stars"
    ] >= 4
).astype("int8")

In [124]:
# 19-7. 평점 피처 집계
# ddof=0을 사용하여 리뷰가 1건이어도 표준편차를 0으로 계산

rating_period_feature_df = (
    sample_rating_review_df
    .groupby(
        [
            "sample_id",
            "period"
        ],
        as_index=False
    )
    .agg(
        rating_review_count=(
            "stars",
            "size"
        ),
        mean_rating=(
            "stars",
            "mean"
        ),
        rating_std=(
            "stars",
            lambda series:
                series.std(ddof=0)
        ),
        low_rating_rate=(
            "is_low_rating",
            "mean"
        ),
        high_rating_rate=(
            "is_high_rating",
            "mean"
        )
    )
)

rating_period_feature_df.head()

,sample_id,period,rating_review_count,mean_rating,rating_std,low_rating_rate,high_rating_rate
0,--Vu3Gux9nPnLcG9yO_HxA_2017,baseline,22,4.590909,1.154402,0.090909,0.909091
1,--Vu3Gux9nPnLcG9yO_HxA_2017,recent,4,4.000000,1.732051,0.250000,0.750000
2,--u09WAjW741FdfkJXxNmg_2017,baseline,19,4.157895,0.874381,0.052632,0.789474
3,--u09WAjW741FdfkJXxNmg_2017,recent,12,4.000000,0.912871,0.083333,0.750000
4,-00kdEIhCt-ODaV4BS-EAg_2012,baseline,12,2.666667,0.942809,0.500000,0.250000


In [125]:
# 19-8. Baseline 평점 피처

baseline_rating_df = (
    rating_period_feature_df[
        rating_period_feature_df[
            "period"
        ].eq("baseline")
    ]
    .drop(
        columns=["period"]
    )
    .rename(
        columns={
            "rating_review_count":
                "baseline_rating_review_count",
            "mean_rating":
                "baseline_mean_rating",
            "rating_std":
                "baseline_rating_std",
            "low_rating_rate":
                "baseline_low_rating_rate",
            "high_rating_rate":
                "baseline_high_rating_rate"
        }
    )
)

# Recent 평점 피처
recent_rating_df = (
    rating_period_feature_df[
        rating_period_feature_df[
            "period"
        ].eq("recent")
    ]
    .drop(
        columns=["period"]
    )
    .rename(
        columns={
            "rating_review_count":
                "recent_rating_review_count",
            "mean_rating":
                "recent_mean_rating",
            "rating_std":
                "recent_rating_std",
            "low_rating_rate":
                "recent_low_rating_rate",
            "high_rating_rate":
                "recent_high_rating_rate"
        }
    )
)

In [126]:
# 19-9. 마스터 코호트와 평점 피처 결합

rating_feature_build_df = (
    rolling_cohort_df[
        [
            "sample_id",
            "user_id",
            "selection_year",
            "churn"
        ]
    ]
    .merge(
        baseline_rating_df,
        on="sample_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        recent_rating_df,
        on="sample_id",
        how="left",
        validate="one_to_one"
    )
)

rating_feature_build_df[
    "mean_rating_change"
] = (
    rating_feature_build_df[
        "recent_mean_rating"
    ]
    - rating_feature_build_df[
        "baseline_mean_rating"
    ]
)

rating_feature_build_df[
    "rating_std_change"
] = (
    rating_feature_build_df[
        "recent_rating_std"
    ]
    - rating_feature_build_df[
        "baseline_rating_std"
    ]
)

rating_feature_build_df[
    "low_rating_rate_increase"
] = (
    rating_feature_build_df[
        "recent_low_rating_rate"
    ]
    - rating_feature_build_df[
        "baseline_low_rating_rate"
    ]
)

rating_feature_build_df[
    "high_rating_rate_decline"
] = (
    rating_feature_build_df[
        "baseline_high_rating_rate"
    ]
    - rating_feature_build_df[
        "recent_high_rating_rate"
    ]
)

In [127]:
# 19-10. 모델용 평점 피처

rating_feature_df = (
    rating_feature_build_df[
        [
            "sample_id",
            "user_id",
            "selection_year",
            "churn",

            "baseline_mean_rating",
            "baseline_rating_std",
            "baseline_low_rating_rate",
            "baseline_high_rating_rate",

            "recent_mean_rating",
            "recent_rating_std",
            "recent_low_rating_rate",
            "recent_high_rating_rate",

            "mean_rating_change",
            "rating_std_change",
            "low_rating_rate_increase",
            "high_rating_rate_decline"
        ]
    ]
    .copy()
)

print(
    "평점 피처 크기:",
    rating_feature_df.shape
)

평점 피처 크기: (21601, 16)


In [128]:
# 19-11. 결측·무한대 검증

rating_feature_columns = [
    column
    for column in rating_feature_df.columns
    if column not in [
        "sample_id",
        "user_id",
        "selection_year",
        "churn"
    ]
]

rating_validation_df = pd.DataFrame({
    "feature": rating_feature_columns,
    "missing_count": [
        rating_feature_df[
            column
        ].isna().sum()
        for column in rating_feature_columns
    ],
    "infinite_count": [
        np.isinf(
            rating_feature_df[
                column
            ]
        ).sum()
        for column in rating_feature_columns
    ]
})

display(rating_validation_df)

assert rating_feature_df[
    "sample_id"
].is_unique

assert len(rating_feature_df) == len(
    rolling_cohort_df
)

assert rating_validation_df[
    "missing_count"
].sum() == 0

assert rating_validation_df[
    "infinite_count"
].sum() == 0

# 평균 평점은 1~5 범위
assert rating_feature_df[
    "baseline_mean_rating"
].between(1, 5).all()

assert rating_feature_df[
    "recent_mean_rating"
].between(1, 5).all()

# 평점 비율은 0~1 범위
rating_rate_columns = [
    "baseline_low_rating_rate",
    "baseline_high_rating_rate",
    "recent_low_rating_rate",
    "recent_high_rating_rate"
]

for column in rating_rate_columns:
    assert rating_feature_df[
        column
    ].between(0, 1).all()

assert (
    rating_feature_df[
        "baseline_rating_std"
    ] >= 0
).all()

assert (
    rating_feature_df[
        "recent_rating_std"
    ] >= 0
).all()

print("롤링 평점 피처 검증 통과")

,feature,missing_count,infinite_count
0,baseline_mean_rating,0,0
1,baseline_rating_std,0,0
2,baseline_low_rating_rate,0,0
3,baseline_high_rating_rate,0,0
4,recent_mean_rating,0,0
5,recent_rating_std,0,0
6,recent_low_rating_rate,0,0
7,recent_high_rating_rate,0,0
8,mean_rating_change,0,0
9,rating_std_change,0,0


롤링 평점 피처 검증 통과


In [129]:
# 19-12. 이탈 여부별 평점 변화 비교

rating_churn_summary_df = (
    rating_feature_df
    .groupby(
        "churn",
        as_index=False
    )
    .agg(
        samples=(
            "sample_id",
            "size"
        ),
        baseline_mean_rating_mean=(
            "baseline_mean_rating",
            "mean"
        ),
        recent_mean_rating_mean=(
            "recent_mean_rating",
            "mean"
        ),
        mean_rating_change_mean=(
            "mean_rating_change",
            "mean"
        ),
        baseline_rating_std_mean=(
            "baseline_rating_std",
            "mean"
        ),
        recent_rating_std_mean=(
            "recent_rating_std",
            "mean"
        ),
        rating_std_change_mean=(
            "rating_std_change",
            "mean"
        ),
        low_rating_rate_increase_mean=(
            "low_rating_rate_increase",
            "mean"
        ),
        high_rating_rate_decline_mean=(
            "high_rating_rate_decline",
            "mean"
        )
    )
)

display(rating_churn_summary_df)

,churn,samples,baseline_mean_rating_mean,recent_mean_rating_mean,mean_rating_change_mean,baseline_rating_std_mean,recent_rating_std_mean,rating_std_change_mean,low_rating_rate_increase_mean,high_rating_rate_decline_mean
0,0,18215,3.772113,3.771187,-0.000926,0.979887,0.908260,-0.071627,0.008571,0.000703
1,1,3386,3.817576,3.790733,-0.026842,0.977955,0.760136,-0.217819,0.031196,0.003832


In [130]:
# 19-13. 리뷰 수 감소와 평점 변화의 상관관계

if "activity_feature_df" not in globals():
    activity_feature_df = pd.read_parquet(
        FEATURE_ROLLING_DIR
        / "activity_features_rolling_v02.parquet"
    )

rating_activity_correlation_df = (
    activity_feature_df[
        [
            "sample_id",
            "review_count_decline_rate"
        ]
    ]
    .merge(
        rating_feature_df[
            [
                "sample_id",
                "mean_rating_change",
                "rating_std_change",
                "low_rating_rate_increase",
                "high_rating_rate_decline"
            ]
        ],
        on="sample_id",
        how="inner",
        validate="one_to_one"
    )
    .drop(
        columns=["sample_id"]
    )
    .corr()
    .round(3)
)

display(rating_activity_correlation_df)

,review_count_decline_rate,mean_rating_change,rating_std_change,low_rating_rate_increase,high_rating_rate_decline
review_count_decline_rate,1.000,-0.029,-0.162,0.098,0.015
mean_rating_change,-0.029,1.000,-0.279,-0.820,-0.861
rating_std_change,-0.162,-0.279,1.000,0.331,0.253
low_rating_rate_increase,0.098,-0.820,0.331,1.000,0.650
high_rating_rate_decline,0.015,-0.861,0.253,0.650,1.000


In [131]:
# 19-14. 평점 피처와 검증 결과 저장

rating_feature_df.to_parquet(
    RATING_FEATURE_PATH,
    index=False
)

rating_validation_df.to_csv(
    RATING_VALIDATION_PATH,
    index=False,
    encoding="utf-8-sig"
)

rating_churn_summary_df.to_csv(
    RATING_CHURN_SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig"
)

rating_activity_correlation_df.to_csv(
    RATING_CORRELATION_PATH,
    encoding="utf-8-sig"
)

print(
    "평점 피처 저장:",
    RATING_FEATURE_PATH
)

print(
    "평점 피처 크기:",
    rating_feature_df.shape
)

평점 피처 저장: C:\Users\playdata2\SKN34-2nd-5Team\data\interim\features_rolling\rating_features_rolling_v02.parquet
평점 피처 크기: (21601, 16)


In [132]:
# 20. 롤링 모델링 데이터셋 생성
# 20-1. 최종 모델링 데이터 저장 경로

PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MODEL_DATASET_PATH = (
    PROCESSED_DIR
    / "modeling_dataset_rolling_v02.parquet"
)

FEATURE_VALIDATION_PATH = (
    REPORT_TABLE_DIR
    / "rolling_model_feature_validation_v02.csv"
)

SPLIT_SUMMARY_PATH = (
    REPORT_TABLE_DIR
    / "rolling_temporal_split_summary_v02.csv"
)

print(
    "최종 모델링 데이터:",
    MODEL_DATASET_PATH
)

최종 모델링 데이터: C:\Users\playdata2\SKN34-2nd-5Team\data\processed\modeling_dataset_rolling_v02.parquet


In [133]:
# 20-2. 피처 파일 경로

feature_file_paths = {
    "activity": (
        FEATURE_ROLLING_DIR
        / "activity_features_rolling_v02.parquet"
    ),
    "interval": (
        FEATURE_ROLLING_DIR
        / "interval_features_rolling_v02.parquet"
    ),
    "business": (
        FEATURE_ROLLING_DIR
        / "business_features_rolling_v02.parquet"
    ),
    "category": (
        FEATURE_ROLLING_DIR
        / "category_features_rolling_v02.parquet"
    ),
    "spatial": (
        FEATURE_ROLLING_DIR
        / "spatial_features_rolling_v02.parquet"
    ),
    "rating": (
        FEATURE_ROLLING_DIR
        / "rating_features_rolling_v02.parquet"
    )
}

feature_file_check_df = pd.DataFrame({
    "feature_group": feature_file_paths.keys(),
    "path": [
        str(path)
        for path in feature_file_paths.values()
    ],
    "exists": [
        path.exists()
        for path in feature_file_paths.values()
    ]
})

display(feature_file_check_df)

missing_files = feature_file_check_df.loc[
    ~feature_file_check_df["exists"],
    "feature_group"
].tolist()

if missing_files:
    raise FileNotFoundError(
        "다음 피처 파일을 찾지 못했습니다: "
        f"{missing_files}"
    )

,feature_group,path,exists
0,activity,C:\Users\playdata2\SKN34-2nd-5Team\data\interi...,True
1,interval,C:\Users\playdata2\SKN34-2nd-5Team\data\interi...,True
2,business,C:\Users\playdata2\SKN34-2nd-5Team\data\interi...,True
3,category,C:\Users\playdata2\SKN34-2nd-5Team\data\interi...,True
4,spatial,C:\Users\playdata2\SKN34-2nd-5Team\data\interi...,True
5,rating,C:\Users\playdata2\SKN34-2nd-5Team\data\interi...,True


In [134]:
# 20-3. 피처 파일 전체 불러오기

feature_dataframes = {
    feature_group: pd.read_parquet(path)
    for feature_group, path
    in feature_file_paths.items()
}

feature_file_summary_df = pd.DataFrame([
    {
        "feature_group": feature_group,
        "rows": len(feature_df),
        "columns": len(feature_df.columns),
        "unique_samples": (
            feature_df["sample_id"].nunique()
        ),
        "duplicate_samples": (
            feature_df["sample_id"]
            .duplicated()
            .sum()
        )
    }
    for feature_group, feature_df
    in feature_dataframes.items()
])

display(feature_file_summary_df)

,feature_group,rows,columns,unique_samples,duplicate_samples
0,activity,21601,18,21601,0
1,interval,21601,16,21601,0
2,business,21601,18,21601,0
3,category,21601,18,21601,0
4,spatial,21601,16,21601,0
5,rating,21601,16,21601,0


In [136]:
# 20-4. 마스터 코호트와 피처 파일의 키·메타데이터 검증

# 모든 피처 파일에 반드시 필요한 컬럼은 sample_id뿐
required_feature_key_columns = {
    "sample_id"
}

# 피처 파일에 존재하는 경우에만 비교할 메타데이터
optional_metadata_columns = [
    "user_id",
    "selection_year",
    "churn"
]

master_metadata_df = (
    rolling_cohort_df[
        [
            "sample_id",
            "user_id",
            "selection_year",
            "churn"
        ]
    ]
    .copy()
)

assert master_metadata_df[
    "sample_id"
].is_unique

for (
    feature_group,
    feature_df
) in feature_dataframes.items():

    # sample_id 존재 여부 확인
    missing_key_columns = (
        required_feature_key_columns
        - set(feature_df.columns)
    )

    assert not missing_key_columns, (
        f"{feature_group} 필수 키 누락: "
        f"{missing_key_columns}"
    )

    # 피처 파일 내부 sample_id 중복 확인
    assert feature_df[
        "sample_id"
    ].is_unique, (
        f"{feature_group}: sample_id 중복"
    )

    # 마스터 코호트와 표본 구성이 같은지 확인
    master_sample_ids = set(
        master_metadata_df[
            "sample_id"
        ]
    )

    feature_sample_ids = set(
        feature_df[
            "sample_id"
        ]
    )

    missing_samples = (
        master_sample_ids
        - feature_sample_ids
    )

    extra_samples = (
        feature_sample_ids
        - master_sample_ids
    )

    assert not missing_samples, (
        f"{feature_group}: "
        f"마스터 표본 {len(missing_samples)}개 누락"
    )

    assert not extra_samples, (
        f"{feature_group}: "
        f"마스터에 없는 표본 {len(extra_samples)}개 존재"
    )

    # user_id·selection_year·churn은
    # 피처 파일에 있을 때만 마스터와 비교
    checked_metadata = []

    for column in optional_metadata_columns:

        if column not in feature_df.columns:
            continue

        metadata_check_df = (
            feature_df[
                [
                    "sample_id",
                    column
                ]
            ]
            .merge(
                master_metadata_df[
                    [
                        "sample_id",
                        column
                    ]
                ],
                on="sample_id",
                how="inner",
                suffixes=(
                    "_feature",
                    "_master"
                ),
                validate="one_to_one"
            )
        )

        feature_values = (
            metadata_check_df[
                f"{column}_feature"
            ]
        )

        master_values = (
            metadata_check_df[
                f"{column}_master"
            ]
        )

        mismatch_count = (
            feature_values.astype(str)
            != master_values.astype(str)
        ).sum()

        assert mismatch_count == 0, (
            f"{feature_group}: "
            f"{column} 불일치 "
            f"{mismatch_count}건"
        )

        checked_metadata.append(column)

    print(
        f"{feature_group}: 검증 통과 | "
        f"표본 {len(feature_df):,}개 | "
        f"확인 메타데이터 {checked_metadata}"
    )

activity: 검증 통과 | 표본 21,601개 | 확인 메타데이터 ['user_id', 'selection_year']
interval: 검증 통과 | 표본 21,601개 | 확인 메타데이터 ['user_id', 'selection_year']
business: 검증 통과 | 표본 21,601개 | 확인 메타데이터 ['user_id', 'selection_year']
category: 검증 통과 | 표본 21,601개 | 확인 메타데이터 ['user_id', 'selection_year', 'churn']
spatial: 검증 통과 | 표본 21,601개 | 확인 메타데이터 ['user_id', 'selection_year', 'churn']
rating: 검증 통과 | 표본 21,601개 | 확인 메타데이터 ['user_id', 'selection_year', 'churn']


In [137]:
# 20-5. 피처 그룹 사이에 같은 이름의 피처가 있는지 확인

identifier_columns = {
    "sample_id",
    "user_id",
    "selection_year",
    "observation_year",
    "target_year",
    "churn",
    "candidate_range_split",
    "recent_range_split",
    "experiment_a_split",
    "experiment_b_split"
}

feature_column_sources = []

for (
    feature_group,
    feature_df
) in feature_dataframes.items():

    for column in feature_df.columns:
        if column not in identifier_columns:
            feature_column_sources.append({
                "feature": column,
                "feature_group": feature_group
            })

feature_column_source_df = pd.DataFrame(
    feature_column_sources
)

duplicate_feature_name_df = (
    feature_column_source_df[
        feature_column_source_df[
            "feature"
        ].duplicated(keep=False)
    ]
    .sort_values(
        [
            "feature",
            "feature_group"
        ]
    )
)

display(duplicate_feature_name_df)

assert duplicate_feature_name_df.empty, (
    "피처 그룹 사이에 중복 피처명이 있습니다."
)

,feature,feature_group


In [138]:
# 20-6. 모델링 데이터의 기본 메타데이터 구성

modeling_dataset_df = (
    rolling_cohort_df[
        [
            "sample_id",
            "user_id",
            "selection_year",
            "churn"
        ]
    ]
    .copy()
)

# 선정연도 y
# 관찰연도 y+1
# 예측 대상연도 y+2
modeling_dataset_df[
    "observation_year"
] = (
    modeling_dataset_df[
        "selection_year"
    ] + 1
)

modeling_dataset_df[
    "target_year"
] = (
    modeling_dataset_df[
        "selection_year"
    ] + 2
)

display(
    modeling_dataset_df.head()
)

,sample_id,user_id,selection_year,churn,observation_year,target_year
0,-KICU2HksIrtaOymb_jqPQ_2009,-KICU2HksIrtaOymb_jqPQ,2009,0,2010,2011
1,-VuSUcCZCbQcdSdF7w9USg_2009,-VuSUcCZCbQcdSdF7w9USg,2009,1,2010,2011
2,-hKniZN2OdshWLHYuj21jQ_2009,-hKniZN2OdshWLHYuj21jQ,2009,0,2010,2011
3,-ju8d9NY3yZyVJCdw5oIUw_2009,-ju8d9NY3yZyVJCdw5oIUw,2009,0,2010,2011
4,-y-R9jOTso_XAjDOrabdFg_2009,-y-R9jOTso_XAjDOrabdFg,2009,0,2010,2011


In [139]:
# 20-7. sample_id를 기준으로 피처 그룹 결합

for (
    feature_group,
    feature_df
) in feature_dataframes.items():

    group_feature_columns = [
        column
        for column in feature_df.columns
        if column not in identifier_columns
    ]

    group_merge_df = feature_df[
        [
            "sample_id",
            *group_feature_columns
        ]
    ]

    before_rows = len(
        modeling_dataset_df
    )

    modeling_dataset_df = (
        modeling_dataset_df
        .merge(
            group_merge_df,
            on="sample_id",
            how="left",
            validate="one_to_one"
        )
    )

    after_rows = len(
        modeling_dataset_df
    )

    assert before_rows == after_rows

    print(
        f"{feature_group} 결합 완료: "
        f"{len(group_feature_columns)}개 피처"
    )

print(
    "통합 모델링 데이터 크기:",
    modeling_dataset_df.shape
)

activity 결합 완료: 15개 피처
interval 결합 완료: 13개 피처
business 결합 완료: 15개 피처
category 결합 완료: 14개 피처
spatial 결합 완료: 12개 피처
rating 결합 완료: 12개 피처
통합 모델링 데이터 크기: (21601, 87)


In [140]:
# 20-8. 모델에 넣지 않을 식별자·시간·정답 컬럼

non_feature_columns = [
    "sample_id",
    "user_id",
    "selection_year",
    "observation_year",
    "target_year",
    "churn"
]

model_feature_columns = [
    column
    for column in modeling_dataset_df.columns
    if column not in non_feature_columns
]

print(
    "전체 표본:",
    len(modeling_dataset_df)
)

print(
    "전체 모델 피처:",
    len(model_feature_columns)
)

print(
    "정답 분포:"
)

display(
    modeling_dataset_df[
        "churn"
    ].value_counts()
)

전체 표본: 21601
전체 모델 피처: 81
정답 분포:


churn
0    18215
1     3386
Name: count, dtype: int64

In [141]:
# 20-9. 미래 정답을 직접 포함하는 피처가 없는지 이름 기준 검증

forbidden_feature_keywords = [
    "target",
    "churn",
    "retained",
    "future"
]

leakage_suspect_columns = [
    column
    for column in model_feature_columns
    if any(
        keyword in column.lower()
        for keyword
        in forbidden_feature_keywords
    )
]

print(
    "타깃 누수 의심 피처:",
    leakage_suspect_columns
)

assert not leakage_suspect_columns

타깃 누수 의심 피처: []


In [142]:
# 20-10. 전체 통합 피처 검증

model_feature_validation_df = pd.DataFrame({
    "feature": model_feature_columns,
    "dtype": [
        str(
            modeling_dataset_df[
                column
            ].dtype
        )
        for column in model_feature_columns
    ],
    "missing_count": [
        modeling_dataset_df[
            column
        ].isna().sum()
        for column in model_feature_columns
    ],
    "infinite_count": [
        np.isinf(
            modeling_dataset_df[
                column
            ]
        ).sum()
        for column in model_feature_columns
    ]
})

display(
    model_feature_validation_df[
        (
            model_feature_validation_df[
                "missing_count"
            ] > 0
        )
        | (
            model_feature_validation_df[
                "infinite_count"
            ] > 0
        )
    ]
)

,feature,dtype,missing_count,infinite_count
19,recent_mean_interval_days,float64,858,0
20,recent_median_interval_days,float64,858,0
21,recent_max_interval_days,float64,858,0
24,mean_interval_increase_days,float64,858,0
25,median_interval_increase_days,float64,858,0
26,max_interval_increase_days,float64,858,0


In [143]:
# 의도적으로 결측을 허용하는 작성 간격 피처

allowed_missing_features = {
    "recent_mean_interval_days",
    "mean_interval_increase_days",
    "recent_median_interval_days",
    "median_interval_increase_days",
    "recent_max_interval_days",
    "max_interval_increase_days"
}

actual_missing_features = set(
    model_feature_validation_df.loc[
        model_feature_validation_df[
            "missing_count"
        ] > 0,
        "feature"
    ]
)

unexpected_missing_features = (
    actual_missing_features
    - allowed_missing_features
)

print(
    "결측 발생 피처:",
    sorted(actual_missing_features)
)

print(
    "예상하지 못한 결측:",
    sorted(unexpected_missing_features)
)

assert not unexpected_missing_features

assert model_feature_validation_df[
    "infinite_count"
].sum() == 0

assert modeling_dataset_df[
    "sample_id"
].is_unique

assert len(modeling_dataset_df) == len(
    rolling_cohort_df
)

print("통합 모델링 데이터 논리 검증 통과")

결측 발생 피처: ['max_interval_increase_days', 'mean_interval_increase_days', 'median_interval_increase_days', 'recent_max_interval_days', 'recent_mean_interval_days', 'recent_median_interval_days']
예상하지 못한 결측: []
통합 모델링 데이터 논리 검증 통과


In [144]:
# 21. 시간 기준 학습 범위 생성
# 21-1. 학습후보 선정연도에 따른 두 가지 실험 범위

# 실험 A
# 사용 가능한 과거 선정연도를 모두 학습에 사용
modeling_dataset_df[
    "experiment_a_split"
] = np.select(
    [
        modeling_dataset_df[
            "selection_year"
        ] <= 2015,

        modeling_dataset_df[
            "selection_year"
        ] == 2016,

        modeling_dataset_df[
            "selection_year"
        ] == 2017
    ],
    [
        "train",
        "validation",
        "test"
    ],
    default="excluded"
)

# 실험 B
# 비교적 데이터가 안정적인 2013~2015년만 학습
modeling_dataset_df[
    "experiment_b_split"
] = np.select(
    [
        modeling_dataset_df[
            "selection_year"
        ].between(
            2013,
            2015
        ),

        modeling_dataset_df[
            "selection_year"
        ] == 2016,

        modeling_dataset_df[
            "selection_year"
        ] == 2017
    ],
    [
        "train",
        "validation",
        "test"
    ],
    default="excluded"
)

In [145]:
# 21-2. 실험별 분할 결과 요약 함수

def build_temporal_split_summary(
    dataframe,
    split_column,
    experiment_name
):
    summary_df = (
        dataframe
        .groupby(
            split_column,
            as_index=False
        )
        .agg(
            samples=(
                "sample_id",
                "size"
            ),
            unique_users=(
                "user_id",
                "nunique"
            ),
            minimum_selection_year=(
                "selection_year",
                "min"
            ),
            maximum_selection_year=(
                "selection_year",
                "max"
            ),
            churn_samples=(
                "churn",
                "sum"
            ),
            churn_rate_pct=(
                "churn",
                "mean"
            )
        )
        .rename(
            columns={
                split_column: "split"
            }
        )
    )

    summary_df[
        "churn_rate_pct"
    ] *= 100

    summary_df.insert(
        0,
        "experiment",
        experiment_name
    )

    return summary_df

In [146]:
experiment_a_summary_df = (
    build_temporal_split_summary(
        dataframe=modeling_dataset_df,
        split_column="experiment_a_split",
        experiment_name=(
            "A_전체_학습후보연도"
        )
    )
)

experiment_b_summary_df = (
    build_temporal_split_summary(
        dataframe=modeling_dataset_df,
        split_column="experiment_b_split",
        experiment_name=(
            "B_2013_2015"
        )
    )
)

temporal_split_summary_df = pd.concat(
    [
        experiment_a_summary_df,
        experiment_b_summary_df
    ],
    ignore_index=True
)

display(
    temporal_split_summary_df
)

,experiment,split,samples,unique_users,minimum_selection_year,maximum_selection_year,churn_samples,churn_rate_pct
0,A_전체_학습후보연도,test,4157,4157,2017,2017,670,16.117392
1,A_전체_학습후보연도,train,13720,8483,2009,2015,2189,15.954810
2,A_전체_학습후보연도,validation,3724,3724,2016,2016,527,14.151450
3,B_2013_2015,excluded,4966,3450,2009,2012,808,16.270640
4,B_2013_2015,test,4157,4157,2017,2017,670,16.117392
5,B_2013_2015,train,8754,6211,2013,2015,1381,15.775645
6,B_2013_2015,validation,3724,3724,2016,2016,527,14.151450


In [147]:
# 21-3. Train → Validation → Test 시간 순서 검증

for split_column in [
    "experiment_a_split",
    "experiment_b_split"
]:
    train_years = (
        modeling_dataset_df.loc[
            modeling_dataset_df[
                split_column
            ].eq("train"),
            "selection_year"
        ]
    )

    validation_years = (
        modeling_dataset_df.loc[
            modeling_dataset_df[
                split_column
            ].eq("validation"),
            "selection_year"
        ]
    )

    test_years = (
        modeling_dataset_df.loc[
            modeling_dataset_df[
                split_column
            ].eq("test"),
            "selection_year"
        ]
    )

    assert not train_years.empty
    assert not validation_years.empty
    assert not test_years.empty

    assert (
        train_years.max()
        < validation_years.min()
    )

    assert (
        validation_years.max()
        < test_years.min()
    )

    print(
        f"{split_column}: "
        "시간 순서 검증 통과"
    )

experiment_a_split: 시간 순서 검증 통과
experiment_b_split: 시간 순서 검증 통과


In [148]:
# 21-4. 두 실험의 Validation·Test 표본 일치 검증

for split_name in [
    "validation",
    "test"
]:
    experiment_a_samples = set(
        modeling_dataset_df.loc[
            modeling_dataset_df[
                "experiment_a_split"
            ].eq(split_name),
            "sample_id"
        ]
    )

    experiment_b_samples = set(
        modeling_dataset_df.loc[
            modeling_dataset_df[
                "experiment_b_split"
            ].eq(split_name),
            "sample_id"
        ]
    )

    assert (
        experiment_a_samples
        == experiment_b_samples
    )

print(
    "두 실험의 Validation·Test "
    "표본 일치 검증 통과"
)

두 실험의 Validation·Test 표본 일치 검증 통과


In [149]:
# 21-5. 최종 모델링 데이터와 검증표 저장

modeling_dataset_df.to_parquet(
    MODEL_DATASET_PATH,
    index=False
)

model_feature_validation_df.to_csv(
    FEATURE_VALIDATION_PATH,
    index=False,
    encoding="utf-8-sig"
)

temporal_split_summary_df.to_csv(
    SPLIT_SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig"
)

print(
    "모델링 데이터 저장:",
    MODEL_DATASET_PATH
)

print(
    "모델링 데이터 크기:",
    modeling_dataset_df.shape
)

print(
    "모델 피처 수:",
    len(model_feature_columns)
)

모델링 데이터 저장: C:\Users\playdata2\SKN34-2nd-5Team\data\processed\modeling_dataset_rolling_v02.parquet
모델링 데이터 크기: (21601, 89)
모델 피처 수: 81
